In [25]:
import pandas as pd
import numpy as np

# 数据加载模块
try:
    # 使用完整路径加载房价训练集和测试集数据
    df_train_price = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_train_price (1).csv')
    df_test_price = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_test_price (1).csv')
    
    # 打印数据加载成功信息和数据维度
    print(f"成功加载训练集，维度: {df_train_price.shape}")
    print(f"成功加载测试集，维度: {df_test_price.shape}")
    
except FileNotFoundError as e:
    # 文件未找到时的错误处理
    print(f"错误：找不到数据文件: {e}")
    print("请检查文件路径是否正确")

# 数据探索分析模块
print("\n--- 训练集前5行 ---")
print(df_train_price.head())

# 查看训练集基本信息，包括列名、数据类型、非空值数量等
print("\n--- 训练集信息 ---")
df_train_price.info()

# 查看数值型列的基本统计信息（计数、均值、标准差、最小值、四分位数等）
print("\n--- 训练集数值统计 ---")
print(df_train_price.describe())

# 如果有测试集，也可以查看测试集的基本信息
if 'df_test_price' in locals():
    print("\n--- 测试集基本信息 ---")
    print(f"测试集维度: {df_test_price.shape}")
    print("\n测试集前3行:")
    print(df_test_price.head(3))

/tmp/ipykernel_89/1862776333.py:7: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_price = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_train_price (1).csv')
/tmp/ipykernel_89/1862776333.py:8: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_price = pd.read_csv('/home/mw/input/midata1852/ruc_Class25Q2_test_price (1).csv')


成功加载训练集，维度: (103871, 55)
成功加载测试集，维度: (34017, 55)

--- 训练集前5行 ---
   城市     区域      板块    环线         Price      房屋户型        所在楼层     建筑面积  \
0   0  109.0   150.0  二至三环  6.194049e+06  2室1厅1厨1卫   中楼层 (共5层)    52.3㎡   
1   0   65.0   299.0  五至六环  4.354153e+06  3室1厅1厨1卫    顶层 (共6层)  127.44㎡   
2   0   62.0   911.0  五至六环  3.321992e+06  3室2厅1厨2卫   低楼层 (共6层)  118.02㎡   
3   0  123.0  1102.0   六环外  7.895656e+06  6室3厅1厨3卫    底层 (共2层)  293.23㎡   
4   0   81.0   295.0  三至四环  1.902960e+06     1房间1卫  中楼层 (共10层)   39.85㎡   

      套内面积     房屋朝向  ...     供水        供暖     供电            燃气费       供热费  \
0      NaN      南 北  ...     民水      集中供暖     民电       2.61元/m³     30元/㎡   
1   123.7㎡      南 北  ...  商水/民水       自采暖  商电/民电       2.61元/m³       NaN   
2  101.95㎡       东南  ...  商水/民水  集中供暖/自采暖  商电/民电       2.61元/m³     30元/㎡   
3  293.23㎡  东 南 西 北  ...     民水       自采暖     民电  2.61-2.63元/m³       NaN   
4   29.94㎡        南  ...  商水/民水  集中供暖/自采暖  商电/民电  2.61-2.63元/m³  30-45元/㎡   

      停车位  停车费用     c

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# 数据预处理模块：特征选择和数据类型清理

# 1.1 自动计算训练集各列缺失率
missing_rates = df_train_price.isna().mean()  # 计算缺失比例（0-1）
high_missing_cols = missing_rates[missing_rates > 0.7].index.tolist()  # 自动筛选缺失率>70%的列
print(f"自动筛选缺失率>70%的列：{high_missing_cols}")

# 1.2 手动指定需删除的列（基于业务理解和数据泄漏考虑）
manual_drop_cols = [
    '客户反馈',         # 数据泄漏（含房价相关反馈）
    '物业办公电话',     # 非特征（联系方式ID，无预测价值）
    '供热费',           # 无预测价值且可能高缺失
]

# 1.3 合并自动筛选和手动指定的删除列
columns_to_drop = list(set(high_missing_cols + manual_drop_cols))
print(f"最终确定删除的列：{columns_to_drop}")
print(f"总共删除 {len(columns_to_drop)} 个特征")

# 在训练集上删除指定列，创建清理后的副本
df_train_price_cleaned = df_train_price.drop(columns=columns_to_drop).copy()

# 在测试集上删除同样的列，确保训练集和测试集特征一致
try:
    df_test_price_cleaned = df_test_price.drop(columns=columns_to_drop).copy()
    print("测试集特征删除成功")
except NameError:
    print("错误：测试集数据未正确加载")

print("已完成基于缺失率分析的特征列筛选")
# 在训练集上删除指定列，创建清理后的副本
df_train_price_cleaned = df_train_price.drop(columns=columns_to_drop).copy()

# 在测试集上删除同样的列，确保训练集和测试集特征一致
try:
    df_test_price_cleaned = df_test_price.drop(columns=columns_to_drop).copy()
except NameError:
    print("错误：测试集数据未正确加载")

print("已完成特征列筛选，移除了数据泄漏、高缺失率和无关特征")

# 数据类型清理：处理带有单位的数值字段

# 清理 '建筑面积' - 移除"㎡"单位并转换为浮点数
df_train_price_cleaned['建筑面积'] = df_train_price_cleaned['建筑面积'].str.replace('㎡', '').astype(float)
df_test_price_cleaned['建筑面积'] = df_test_price_cleaned['建筑面积'].str.replace('㎡', '').astype(float)

# 清理 '套内面积' - 同样移除单位并转换类型
df_train_price_cleaned['套内面积'] = df_train_price_cleaned['套内面积'].str.replace('㎡', '').astype(float)
df_test_price_cleaned['套内面积'] = df_test_price_cleaned['套内面积'].str.replace('㎡', '').astype(float)

print("已完成面积字段的单位清理和类型转换")

# 数据集分割：分离特征和目标变量

# 分离特征(X)和目标变量(y)
y = df_train_price_cleaned['Price']  # 目标变量：房价
X = df_train_price_cleaned.drop(columns=['Price'])  # 特征矩阵：移除目标变量

# 将数据集按80/20比例分割为训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=111  # 设置随机种子确保结果可复现
)

# 创建独立的数据副本，避免SettingWithCopyWarning警告
X_train = X_train.copy()
X_val = X_val.copy()

# 为测试集创建独立副本
X_test_price = df_test_price_cleaned.copy()

print(f"数据集分割完成 - 训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test_price.shape}")

# 测试集ID列处理

# 删除测试集中的ID列（如果存在），避免数据泄漏
if 'ID' in X_test_price.columns:
    X_test_price = X_test_price.drop(columns=['ID'])
    print("已删除测试集中的ID列")

# 缺失值处理模块

# 识别数值型和类别型特征
numerical_features = X_train.select_dtypes(include=['float64', 'int64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

print(f"\n缺失值填充策略:")
print(f"将使用 '中位数' 填充 {len(numerical_features)} 个数值型特征")
print(f"将使用 'Unknown' 填充 {len(categorical_features)} 个类别型特征")

# 填充数值型特征 - 使用训练集的中位数（避免数据泄漏）
for col in numerical_features:
    # 从训练集计算中位数
    median_value = X_train[col].median()
    
    # 用训练集中位数填充所有三个数据集
    X_train[col] = X_train[col].fillna(median_value)
    X_val[col] = X_val[col].fillna(median_value)
    
    # 确保测试集中也有这一列才进行填充
    if col in X_test_price.columns:
        X_test_price[col] = X_test_price[col].fillna(median_value)

print("已完成数值型特征的缺失值填充")

# 填充类别型特征 - 使用"Unknown"标记缺失值
fill_value = "Unknown" 

for col in categorical_features:
    # 填充所有三个数据集的类别型特征
    X_train[col] = X_train[col].fillna(fill_value)
    X_val[col] = X_val[col].fillna(fill_value)
    
    # 确保测试集中也有这一列才进行填充
    if col in X_test_price.columns:
        X_test_price[col] = X_test_price[col].fillna(fill_value)

print("已完成类别型特征的缺失值填充")

# 类别特征分析：检查特征基数

# 重新识别object类型的列（填充后可能有变化）
categorical_features = X_train.select_dtypes(include=['object']).columns

print(f"\n特征分析:")
print(f"在 X_train 中找到了 {len(categorical_features)} 个 object (文本) 类型的特征。")

# 计算并分析类别特征的基数（唯一值数量）
print("\n--- 类别特征的基数 (唯一值数量) 检查 ---")
cardinality = X_train[categorical_features].nunique()

# 按照唯一值数量从低到高排序，便于分析
print(cardinality.sort_values())

# 输出当前数据维度
print(f"\n数据预处理完成 - X_train 当前维度: {X_train.shape}")

自动筛选缺失率>70%的列：['别墅类型', '抵押信息', '户型介绍', '环线位置', '供暖']
最终确定删除的列：['户型介绍', '别墅类型', '供暖', '物业办公电话', '供热费', '客户反馈', '环线位置', '抵押信息']
总共删除 8 个特征
测试集特征删除成功
已完成基于缺失率分析的特征列筛选
已完成特征列筛选，移除了数据泄漏、高缺失率和无关特征
已完成面积字段的单位清理和类型转换
数据集分割完成 - 训练集: (83096, 46), 验证集: (20775, 46), 测试集: (34017, 47)
已删除测试集中的ID列

缺失值填充策略:
将使用 '中位数' 填充 14 个数值型特征
将使用 'Unknown' 填充 32 个类别型特征
已完成数值型特征的缺失值填充
已完成类别型特征的缺失值填充

特征分析:
在 X_train 中找到了 32 个 object (文本) 类型的特征。

--- 类别特征的基数 (唯一值数量) 检查 ---
产权所属             2
配备电梯             3
供电               4
供水               4
房屋年限             4
装修情况             5
建筑结构             8
环线              12
交易权属            15
建筑结构_comm       16
房屋用途            21
房屋优势            28
房屋朝向           162
产权描述           167
楼栋总数           209
物业类别           227
绿 化 率          233
停车费用           248
所在楼层           254
燃气费            261
梯户比例           315
房屋户型           362
建筑年代           723
物 业 费         1011
交易时间          1585
房屋总数          1723
物业公司          1850
开发商           1984
上次交易          8036
交

In [27]:
print("\n--- 3.1 文本存在性特征提取 ---")

# 定义需处理的文本列
text_cols = ['交通出行', '周边配套', '核心卖点']
new_features = [f'Has_{col}' for col in text_cols]  # 明确新特征列表，避免冗余

# 对每个数据集（训练/验证/测试）处理，确保1:1生成新特征
for df in [X_train, X_val, X_test_price]:
    # 仅处理数据集中存在的文本列，避免无效操作
    valid_cols = [col for col in text_cols if col in df.columns]
    for col in valid_cols:
        # 直接通过向量化操作生成二元特征（效率高，且仅新增1列/原始列）
        df[f'Has_{col}'] = (df[col] != 'Unknown').astype(int)

print("文本存在性特征提取完成")
print(f"新增特征: {new_features}")
print(f"X_train 维度: {X_train.shape}")


--- 3.1 文本存在性特征提取 ---
文本存在性特征提取完成
新增特征: ['Has_交通出行', 'Has_周边配套', 'Has_核心卖点']
X_train 维度: (83096, 49)


In [28]:
import re
import numpy as np
import pandas as pd

print("--- 步骤 3.2 (最终合并版) ---")

# =============================================
# 1. 文本解析函数组 - 用于处理文本数值提取
# =============================================

def parse_build_year(text_data):
    """解析建筑年代"""
    if not isinstance(text_data, str):
        return np.nan
    
    numbers = re.findall(r'(\d{4})', text_data)
    
    if len(numbers) == 0:
        return np.nan
    elif len(numbers) == 1:
        return float(numbers[0])
    else:
        first_year = float(numbers[0])
        last_year = float(numbers[-1])
        return (first_year + last_year) / 2

def parse_property_fee(text_data):
    """解析物业费"""
    if not isinstance(text_data, str):
        return np.nan
    
    numbers = re.findall(r'(\d+\.?\d*)', text_data)
    numbers = [n for n in numbers if n] 
    
    if len(numbers) == 0:
        return np.nan
    elif len(numbers) == 1:
        return float(numbers[0])
    else:
        first_num = float(numbers[0])
        last_num = float(numbers[-1])
        return (first_num + last_num) / 2

def parse_total_units(text_data):
    """解析房屋总数"""
    if not isinstance(text_data, str):
        return np.nan
    
    match = re.search(r'(\d+)', text_data)
    if match:
        return float(match.group(1))
    else:
        return np.nan

# =============================================
# 2. 文本字段处理 - 使用统一的处理模式
# =============================================

# 定义处理配置：列名 -> (解析函数, 新列名)
text_processing_config = {
    '建筑年代': (parse_build_year, 'House_Age'),
    '物 业 费': (parse_property_fee, 'Property_Fee'), 
    '房屋总数': (parse_total_units, 'Total_Units')
}

# 统一处理所有文本字段
for col_name, (parse_func, new_col_name) in text_processing_config.items():
    print(f"\n--- 处理 '{col_name}' ---")
    
    # 应用解析函数
    if col_name == '建筑年代':
        # 建筑年代特殊处理：解析后计算房龄
        year_extracted_train = X_train[col_name].apply(parse_func)
        year_extracted_val = X_val[col_name].apply(parse_func)
        year_extracted_test = X_test_price[col_name].apply(parse_func)
        
        X_train[new_col_name] = 2025 - year_extracted_train
        X_val[new_col_name] = 2025 - year_extracted_val
        X_test_price[new_col_name] = 2025 - year_extracted_test
    else:
        # 其他字段直接应用
        X_train[new_col_name] = X_train[col_name].apply(parse_func)
        X_val[new_col_name] = X_val[col_name].apply(parse_func)
        X_test_price[new_col_name] = X_test_price[col_name].apply(parse_func)
    
    # 填充NaN值
    median_val = X_train[new_col_name].median()
    X_train[new_col_name] = X_train[new_col_name].fillna(median_val)
    X_val[new_col_name] = X_val[new_col_name].fillna(median_val)
    X_test_price[new_col_name] = X_test_price[new_col_name].fillna(median_val)
    
    print(f"已创建 '{new_col_name}'，使用中位数 {median_val:.2f} 填充 NaN")

# =============================================
# 3. 日期字段处理 - 保持原有逻辑
# =============================================

print("\n--- 处理日期字段 ---")

# 处理交易时间
train_dates = pd.to_datetime(X_train['交易时间'], errors='coerce')
val_dates = pd.to_datetime(X_val['交易时间'], errors='coerce')
test_dates = pd.to_datetime(X_test_price['交易时间'], errors='coerce')

X_train['Transaction_Year'] = train_dates.dt.year
X_train['Transaction_Month'] = train_dates.dt.month
X_val['Transaction_Year'] = val_dates.dt.year
X_val['Transaction_Month'] = val_dates.dt.month
X_test_price['Transaction_Year'] = test_dates.dt.year
X_test_price['Transaction_Month'] = test_dates.dt.month

# 处理上次交易
last_trans_year_train = pd.to_datetime(X_train['上次交易'], errors='coerce').dt.year
last_trans_year_val = pd.to_datetime(X_val['上次交易'], errors='coerce').dt.year
last_trans_year_test = pd.to_datetime(X_test_price['上次交易'], errors='coerce').dt.year

X_train['Years_Since_Last_Transaction'] = X_train['Transaction_Year'] - last_trans_year_train
X_val['Years_Since_Last_Transaction'] = X_val['Transaction_Year'] - last_trans_year_val
X_test_price['Years_Since_Last_Transaction'] = X_test_price['Transaction_Year'] - last_trans_year_test

print("已创建日期相关特征")

# =============================================
# 4. 填充日期特征的NaN值
# =============================================

print("\n--- 填充日期特征的 NaN ---")

# 填充 Transaction_Year 和 Transaction_Month
for col in ['Transaction_Year', 'Transaction_Month']:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test_price[col] = X_test_price[col].fillna(median_val)
    print(f"已使用中位数 {median_val:.0f} 填充 '{col}'")

# 填充 Years_Since_Last_Transaction (使用0填充，代表新房)
X_train['Years_Since_Last_Transaction'] = X_train['Years_Since_Last_Transaction'].fillna(0)
X_val['Years_Since_Last_Transaction'] = X_val['Years_Since_Last_Transaction'].fillna(0)
X_test_price['Years_Since_Last_Transaction'] = X_test_price['Years_Since_Last_Transaction'].fillna(0)
print("已使用 0 填充 'Years_Since_Last_Transaction' (新房假设)")

# =============================================
# 5. 检查结果
# =============================================

print("\n--- 处理完成，检查结果 ---")

print("\n文本特征 (前3行):")
text_features = ['House_Age', 'Property_Fee', 'Total_Units']
print(X_train[text_features].head(3))

print("\n日期特征 (前3行):")
date_features = ['Transaction_Year', 'Transaction_Month', 'Years_Since_Last_Transaction']
print(X_train[date_features].head(3))

print(f"\n数据类型检查:")
for feature in text_features + date_features:
    dtype = X_train[feature].dtype
    print(f"  {feature}: {dtype}")

--- 步骤 3.2 (最终合并版) ---

--- 处理 '建筑年代' ---
已创建 'House_Age'，使用中位数 16.50 填充 NaN

--- 处理 '物 业 费' ---
已创建 'Property_Fee'，使用中位数 1.90 填充 NaN

--- 处理 '房屋总数' ---
已创建 'Total_Units'，使用中位数 1371.00 填充 NaN

--- 处理日期字段 ---
已创建日期相关特征

--- 填充日期特征的 NaN ---
已使用中位数 2024 填充 'Transaction_Year'
已使用中位数 7 填充 'Transaction_Month'
已使用 0 填充 'Years_Since_Last_Transaction' (新房假设)

--- 处理完成，检查结果 ---

文本特征 (前3行):
       House_Age  Property_Fee  Total_Units
20486       23.5         0.600        432.0
53134       16.5         1.900       2494.0
8277        36.0         1.895       2176.0

日期特征 (前3行):
       Transaction_Year  Transaction_Month  Years_Since_Last_Transaction
20486              2024                  7                           0.0
53134              2023                 11                           6.0
8277               2024                  8                           9.0

数据类型检查:
  House_Age: float64
  Property_Fee: float64
  Total_Units: float64
  Transaction_Year: int64
  Transaction_Month: int64
  Yea

In [29]:
import re
import numpy as np
import pandas as pd # 每个代码框独立导入基础库，避免依赖

# ------------------------------------------------------------------------------
# 模块1：量化“房屋户型”（生成4个新特征）
# ------------------------------------------------------------------------------
print("--- 步骤 3.3.1：开始高级量化 '房屋户型' ---")

# 1. 定义房屋户型解析函数
def parse_layout(text_data):
    if not isinstance(text_data, str):
        return (np.nan, np.nan, np.nan, np.nan)
    
    # 提取室/房间、厅、厨、卫的数量
    room_match = re.search(r'(\d+)(?:室|房间)', text_data)
    num_rooms = int(room_match.group(1)) if room_match else 0
    
    living_match = re.search(r'(\d+)厅', text_data)
    num_living = int(living_match.group(1)) if living_match else 0
    
    kitchen_match = re.search(r'(\d+)厨', text_data)
    num_kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
    
    bath_match = re.search(r'(\d+)卫', text_data)
    num_bath = int(bath_match.group(1)) if bath_match else 0

    # 全为0视为无效数据，返回NaN
    if num_rooms == 0 and num_living == 0 and num_kitchen == 0 and num_bath == 0:
         return (np.nan, np.nan, np.nan, np.nan)
    
    return (num_rooms, num_living, num_kitchen, num_bath)

# 2. 应用函数生成新列并合并
new_cols_train = X_train['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
new_cols_val = X_val['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))
new_cols_test = X_test_price['房屋户型'].apply(lambda x: pd.Series(parse_layout(x), index=['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']))

X_train = pd.concat([X_train, new_cols_train], axis=1)
X_val = pd.concat([X_val, new_cols_val], axis=1)
X_test_price = pd.concat([X_test_price, new_cols_test], axis=1)
print("已成功创建 4 个新特征：'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths'。")

# 3. 用训练集中位数填充NaN
print(f"\n--- 步骤 3.3.2：为 4 个新户型特征填充 NaN ---")
new_layout_cols = ['Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths']
for col in new_layout_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test_price[col] = X_test_price[col].fillna(median_val)
    print(f"已使用中位数 {median_val:.0f} 填充 '{col}'。")

# 4. 验证结果
print("\n--- 新的户型特征 (前5行) ---")
print(X_train[new_layout_cols].head())


# ------------------------------------------------------------------------------
# 模块2：量化“所在楼层”（生成3个新特征）
# ------------------------------------------------------------------------------
print("\n--- 步骤 3.4.1 (最终修正版)：开始高级量化 '所在楼层' ---")
print("采纳策略：将'低/中/高'直接映射为比例，生成交互特征。")

# 1. 定义总楼层数解析函数
def parse_total_floors(text_data):
    if not isinstance(text_data, str): return 1.0 # 默认至少1层
    match = re.search(r'共(\d+)层', text_data)
    if match:
        floors = float(match.group(1))
        return max(floors, 1.0) # 确保不小于1
    else:
        return 1.0 # 默认值

# 2. 生成“总楼层数”特征并合并
X_train['Total_Floors'] = X_train['所在楼层'].apply(parse_total_floors)
X_val['Total_Floors'] = X_val['所在楼层'].apply(parse_total_floors)
X_test_price['Total_Floors'] = X_test_price['所在楼层'].apply(parse_total_floors)
print("已成功创建 'Total_Floors' 特征。")

# 3. 定义楼层比例映射字典
floor_ratio_mapping = {
    'Unknown': np.nan,
    '地下室': 0.0,
    '底层': 0.0,
    '低楼层': 0.25,
    '中楼层': 0.5,
    '高楼层': 0.75,
    '顶层': 1.0
}

# 4. 提取楼层类别并映射为比例
def extract_floor_category(text_data):
    if not isinstance(text_data, str): return 'Unknown'
    if '地下室' in text_data: return '地下室'
    if '顶层' in text_data: return '顶层'
    if '高楼层' in text_data: return '高楼层'
    if '中楼层' in text_data: return '中楼层'
    if '低楼层' in text_data: return '低楼层'
    if '底层' in text_data: return '底层'
    return 'Unknown' 

X_train['Floor_Ratio'] = X_train['所在楼层'].apply(extract_floor_category).map(floor_ratio_mapping)
X_val['Floor_Ratio'] = X_val['所在楼层'].apply(extract_floor_category).map(floor_ratio_mapping)
X_test_price['Floor_Ratio'] = X_test_price['所在楼层'].apply(extract_floor_category).map(floor_ratio_mapping)
print("已成功创建 'Floor_Ratio' 特征。")

# 5. 用训练集中位数填充Floor_Ratio的NaN
print(f"\n--- 步骤 3.4.2 (续)：为 'Floor_Ratio' 填充 NaN ---")
median_floor_ratio = X_train['Floor_Ratio'].median()
X_train['Floor_Ratio'] = X_train['Floor_Ratio'].fillna(median_floor_ratio)
X_val['Floor_Ratio'] = X_val['Floor_Ratio'].fillna(median_floor_ratio)
X_test_price['Floor_Ratio'] = X_test_price['Floor_Ratio'].fillna(median_floor_ratio)
print(f"已使用中位数 {median_floor_ratio:.2f} 填充 'Floor_Ratio'。")

# 6. 生成交互特征“预估楼层”
X_train['Estimated_Floor'] = X_train['Total_Floors'] * X_train['Floor_Ratio']
X_val['Estimated_Floor'] = X_val['Total_Floors'] * X_val['Floor_Ratio']
X_test_price['Estimated_Floor'] = X_test_price['Total_Floors'] * X_test_price['Floor_Ratio']
print("已成功创建 'Estimated_Floor' 交互特征！")

# 7. 验证结果
new_floor_cols = ['所在楼层', 'Total_Floors', 'Floor_Ratio', 'Estimated_Floor']
print("\n--- 新的楼层特征 (10个随机样本) ---")
print(X_train[new_floor_cols].sample(10, random_state=114))

--- 步骤 3.3.1：开始高级量化 '房屋户型' ---
已成功创建 4 个新特征：'Num_Rooms', 'Num_LivingRooms', 'Num_Kitchens', 'Num_Baths'。

--- 步骤 3.3.2：为 4 个新户型特征填充 NaN ---
已使用中位数 3 填充 'Num_Rooms'。
已使用中位数 2 填充 'Num_LivingRooms'。
已使用中位数 1 填充 'Num_Kitchens'。
已使用中位数 1 填充 'Num_Baths'。

--- 新的户型特征 (前5行) ---
       Num_Rooms  Num_LivingRooms  Num_Kitchens  Num_Baths
20486        2.0              1.0           1.0        1.0
53134        3.0              2.0           1.0        2.0
8277         1.0              1.0           1.0        1.0
45156        2.0              1.0           1.0        1.0
99942        3.0              2.0           1.0        1.0

--- 步骤 3.4.1 (最终修正版)：开始高级量化 '所在楼层' ---
采纳策略：将'低/中/高'直接映射为比例，生成交互特征。
已成功创建 'Total_Floors' 特征。
已成功创建 'Floor_Ratio' 特征。

--- 步骤 3.4.2 (续)：为 'Floor_Ratio' 填充 NaN ---
已使用中位数 0.50 填充 'Floor_Ratio'。
已成功创建 'Estimated_Floor' 交互特征！

--- 新的楼层特征 (10个随机样本) ---
             所在楼层  Total_Floors  Floor_Ratio  Estimated_Floor
59648  中楼层 (共34层)          34.0         0.50            17.00
23

In [30]:
import re
import numpy as np
import pandas as pd # 每个代码框独立导入基础库，避免依赖

# ------------------------------------------------------------------------------
# 模块3：量化“梯户比例”（生成3个新特征，中文数字映射扩大到200）
# ------------------------------------------------------------------------------
print("\n--- 步骤 3.5.1 ：开始高级量化 '梯户比例' ---")

# 1. 定义中文数字到阿拉伯数字的映射（扩大到200）
chinese_num_map = {}
# 个位数：一(1)~九(9)
ge_chars = ['一','二','三','四','五','六','七','八','九']
for i in range(9):
    chinese_num_map[ge_chars[i]] = i + 1

# 十位数：十(10)~九十九(99)
shi_char = '十'
chinese_num_map[shi_char] = 10
# 十一(11)~十九(19)
for i in range(1, 10):
    chinese_num_map[f'{shi_char}{ge_chars[i-1]}'] = 10 + i
# 二十(20)~九十九(99)
for i in range(2, 10): # 二十、三十...九十
    shi_prefix = ge_chars[i-1]
    chinese_num_map[f'{shi_prefix}{shi_char}'] = i * 10
    # 二十一(21)~九十九(99)（非整十）
    for j in range(1, 10):
        chinese_num_map[f'{shi_prefix}{shi_char}{ge_chars[j-1]}'] = i*10 + j

# 百位数：一百(100)~二百(200)
bai_char = '百'
chinese_num_map[f'一{bai_char}'] = 100
# 一百零一(101)~一百零九(109)
for i in range(1, 10):
    chinese_num_map[f'一{bai_char}零{ge_chars[i-1]}'] = 100 + i
# 一百一十(110)~一百九十九(199)
for i in range(10, 100):
    shi_digit = i // 10 # 十位数字（1~9）
    ge_digit = i % 10   # 个位数字（0~9）
    if ge_digit == 0: # 一百一十、一百二十...一百九十
        chinese_num_map[f'一{bai_char}{ge_chars[shi_digit-1]}{shi_char}'] = 100 + i
    else: # 一百一十一(111)~一百九十九(199)
        chinese_num_map[f'一{bai_char}{ge_chars[shi_digit-1]}{shi_char}{ge_chars[ge_digit-1]}'] = 100 + i
# 二百(200)
chinese_num_map[f'二{bai_char}'] = 200

# 生成中文数字正则表达式（匹配所有key）
chinese_num_regex = f"([{ ''.join(chinese_num_map.keys()) }])"

# 2. 定义梯户比例解析函数
def parse_elevator_ratio(text_data):
    if not isinstance(text_data, str):
        return (np.nan, np.nan)

    # 提取电梯数（优先中文数字，再阿拉伯数字）
    elevator_match = re.search(chinese_num_regex + '梯', text_data)
    if elevator_match:
        num_elevators = chinese_num_map.get(elevator_match.group(1))
    else:
        elevator_match_num = re.search(r'(\d+)梯', text_data)
        num_elevators = float(elevator_match_num.group(1)) if elevator_match_num else np.nan

    # 提取户数（优先中文数字，再阿拉伯数字）
    household_match = re.search(chinese_num_regex + '户', text_data)
    if household_match:
        num_households = chinese_num_map.get(household_match.group(1))
    else:
        household_match_num = re.search(r'(\d+)户', text_data)
        num_households = float(household_match_num.group(1)) if household_match_num else np.nan
            
    return (num_elevators, num_households)

# 3. 应用函数生成新列并合并
new_cols_train = X_train['梯户比例'].apply(lambda x: pd.Series(parse_elevator_ratio(x), index=['Num_Elevators', 'Num_Households']))
new_cols_val = X_val['梯户比例'].apply(lambda x: pd.Series(parse_elevator_ratio(x), index=['Num_Elevators', 'Num_Households']))
new_cols_test = X_test_price['梯户比例'].apply(lambda x: pd.Series(parse_elevator_ratio(x), index=['Num_Elevators', 'Num_Households']))

X_train = pd.concat([X_train, new_cols_train], axis=1)
X_val = pd.concat([X_val, new_cols_val], axis=1)
X_test_price = pd.concat([X_test_price, new_cols_test], axis=1)
print("已成功创建 'Num_Elevators' 和 'Num_Households' 特征。")

# 4. 用训练集中位数填充NaN（修正版）
print(f"\n--- 步骤 3.5.2：为新梯户特征填充 NaN ---")
new_elevator_cols = ['Num_Elevators', 'Num_Households']
for col in new_elevator_cols:
    # 确保列是数值类型（非数值转为NaN）
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    # 计算中位数（强制转为标量）
    median_val = X_train[col].median()
    # 处理全为NaN的极端情况（给一个默认值，如2）
    if pd.isna(median_val):
        median_val = 2  # 可根据业务调整默认值
    # 确保是Python标量（非Series）
    median_val = float(median_val)
    
    # 填充训练集、验证集、测试集
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce').fillna(median_val)
    X_test_price[col] = pd.to_numeric(X_test_price[col], errors='coerce').fillna(median_val)
    
    # 安全打印
    print(f"已使用中位数 {int(median_val)} 填充 '{col}'。")  # 用int避免float格式化问题
    
# 5. 生成梯户比特征（避免除以0）
X_train['Elevator_Ratio'] = X_train['Num_Elevators'] / X_train['Num_Households'].clip(lower=1)
X_val['Elevator_Ratio'] = X_val['Num_Elevators'] / X_val['Num_Households'].clip(lower=1)
X_test_price['Elevator_Ratio'] = X_test_price['Num_Elevators'] / X_test_price['Num_Households'].clip(lower=1)
print("已成功创建 'Elevator_Ratio' 交互特征！")

# 6. 验证结果
check_cols = ['梯户比例', 'Num_Elevators', 'Num_Households', 'Elevator_Ratio']
print("\n--- 新的梯户特征 (5个随机样本) ---")
print(X_train[check_cols].sample(5, random_state=115))


--- 步骤 3.5.1 ：开始高级量化 '梯户比例' ---
已成功创建 'Num_Elevators' 和 'Num_Households' 特征。

--- 步骤 3.5.2：为新梯户特征填充 NaN ---
已使用中位数 1 填充 'Num_Elevators'。
已使用中位数 4 填充 'Num_Households'。
已成功创建 'Elevator_Ratio' 交互特征！

--- 新的梯户特征 (5个随机样本) ---
       梯户比例  Num_Elevators  Num_Households  Elevator_Ratio
49390  两梯四户            1.0             4.0           0.250
809    两梯八户            1.0             8.0           0.125
24733  两梯八户            1.0             8.0           0.125
68121  一梯两户            1.0             4.0           0.250
19833  一梯两户            1.0             4.0           0.250


In [31]:

import re
import numpy as np
import pandas as pd  # 每个代码框独立导入基础库，避免依赖


# ------------------------------------------------------------------------------
# 第一步：提取共用解析函数（带存在性检查，避免重复定义）
# ------------------------------------------------------------------------------
# 1. 共用函数1：解析费用/比例类文本（绿化率、燃气费共用）
try:
    parse_property_fee
except NameError:
    print("重新定义 parse_property_fee 函数...")
    def parse_property_fee(text_data):
        if not isinstance(text_data, str): return np.nan
        numbers = re.findall(r'(\d+\.?\d*)', text_data)
        numbers = [n for n in numbers if n] 
        if len(numbers) == 0: return np.nan
        elif len(numbers) == 1: return float(numbers[0])
        else: return (float(numbers[0]) + float(numbers[-1])) / 2

# 2. 共用函数2：解析数量类文本（楼栋总数专用）
try:
    parse_total_units
except NameError:
    print("重新定义 parse_total_units 函数...")
    def parse_total_units(text_data):
        if not isinstance(text_data, str): return np.nan
        match = re.search(r'(\d+)', text_data)
        if match: return float(match.group(1))
        else: return np.nan


# ------------------------------------------------------------------------------
# 第二步：提取单输出特征量化通用流程（统一处理逻辑，避免重复代码）
# ------------------------------------------------------------------------------
def process_single_feature(
    train_df, val_df, test_df,
    target_col, parse_func, new_col,
    extra_calc=None, median_format=".2f", sample_random_state=111
):
    """
    单输出特征量化通用函数（不改变原功能）
    extra_calc: 特征生成后的额外计算（如 lambda x: x/100），默认None
    median_format: 中位数打印格式（如".2f"保留两位小数，".0f"取整）
    sample_random_state: 验证样本的随机种子
    """
    # 1. 打印开始信息
    print(f"\n--- 步骤 3.6：开始高级量化 '{target_col}' ---")
    
    # 2. 生成新特征
    train_new = train_df[target_col].apply(parse_func)
    val_new = val_df[target_col].apply(parse_func)
    test_new = test_df[target_col].apply(parse_func)
    
    # 执行额外计算（如绿化率转比例）
    if extra_calc:
        train_new = extra_calc(train_new)
        val_new = extra_calc(val_new)
        test_new = extra_calc(test_new)
    
    # 合并新特征到数据集
    train_df[new_col] = train_new
    val_df[new_col] = val_new
    test_df[new_col] = test_new
    print(f"已成功创建 '{new_col}' ({target_col}) 特征。")
    
    # 3. 用训练集中位数填充NaN
    print(f"\n--- 步骤 3.6 ：为 '{new_col}' 填充 NaN ---")
    median_val = train_df[new_col].median()
    train_df[new_col] = train_df[new_col].fillna(median_val)
    val_df[new_col] = val_df[new_col].fillna(median_val)
    test_df[new_col] = test_df[new_col].fillna(median_val)
    print(f"已使用中位数 {median_val:{median_format}} 填充 '{new_col}' 的 NaN 值。")
    
    # 4. 验证结果（打印5个随机样本）
    check_cols = [target_col, new_col]
    print(f"\n--- 新的{target_col}特征 (5个随机样本) ---")
    print(train_df[check_cols].sample(5, random_state=sample_random_state))


# ------------------------------------------------------------------------------
# 第三步：调用通用函数处理三个特征（功能与原代码完全一致）
# ------------------------------------------------------------------------------
# 1. 处理“绿 化 率”（需转比例，中位数保留两位小数，随机种子116）
process_single_feature(
    X_train, X_val, X_test_price,
    target_col="绿 化 率",
    parse_func=parse_property_fee,
    new_col="Greening_Rate",
    extra_calc=lambda x: x / 100.0,  # 转比例（原功能）
    sample_random_state=116
)

# 2. 处理“燃气费”（无额外计算，中位数保留两位小数，随机种子117）
process_single_feature(
    X_train, X_val, X_test_price,
    target_col="燃气费",
    parse_func=parse_property_fee,
    new_col="Gas_Fee",
    sample_random_state=117
)

# 3. 处理“楼栋总数”（用parse_total_units，中位数取整，随机种子124）
process_single_feature(
    X_train, X_val, X_test_price,
    target_col="楼栋总数",
    parse_func=parse_total_units,
    new_col="Total_Buildings",
    median_format=".0f",  # 整数格式化（原功能）
    sample_random_state=124
)


--- 步骤 3.6：开始高级量化 '绿 化 率' ---
已成功创建 'Greening_Rate' (绿 化 率) 特征。

--- 步骤 3.6 ：为 'Greening_Rate' 填充 NaN ---
已使用中位数 0.34 填充 'Greening_Rate' 的 NaN 值。

--- 新的绿 化 率特征 (5个随机样本) ---
         绿 化 率  Greening_Rate
2989       35%           0.35
35234      35%           0.35
95433      30%           0.30
97868      57%           0.57
27546  Unknown           0.34

--- 步骤 3.6：开始高级量化 '燃气费' ---
已成功创建 'Gas_Fee' (燃气费) 特征。

--- 步骤 3.6 ：为 'Gas_Fee' 填充 NaN ---
已使用中位数 2.61 填充 'Gas_Fee' 的 NaN 值。

--- 新的燃气费特征 (5个随机样本) ---
            燃气费  Gas_Fee
2864   2.61元/m³     2.61
49527   Unknown     2.61
53803   Unknown     2.61
11681  2.61元/m³     2.61
6343   2.61元/m³     2.61

--- 步骤 3.6：开始高级量化 '楼栋总数' ---
已成功创建 'Total_Buildings' (楼栋总数) 特征。

--- 步骤 3.6 ：为 'Total_Buildings' 填充 NaN ---
已使用中位数 15 填充 'Total_Buildings' 的 NaN 值。

--- 新的楼栋总数特征 (5个随机样本) ---
          楼栋总数  Total_Buildings
295         1栋              1.0
87834  Unknown             15.0
97247       1栋              1.0
29800      10栋             10.0
56300  U

In [32]:
import re
import numpy as np
import pandas as pd # 每个代码框独立导入基础库，避免依赖

# ------------------------------------------------------------------------------
# 模块：量化“停车费用”（生成1个新特征，硬编码特殊情况）
# ------------------------------------------------------------------------------
print("\n--- 步骤 3.7.1 (硬编码特殊情况版)：开始高级量化 '停车费用' ---")
print("直接映射指定特殊情况到期望值。")

# 1. 硬编码特殊情况的停车费映射
hardcoded_parking_fees = {
    "一元钱一小时，单次24小时内最高12元一次": 12.0 * 30, 
    "小区没有停车费": 0.0,
    "无固定车位不收费": 0.0,
    "售价:30万/位；租价450元/位/月": 450.0,
    "每小时2元每个月70元": 70.0, 
    "每小时2元/位 , 每个月50元/位": 50.0, 
    "露天250元/月/位，3元/时/位；室内500元/月/位，3元/时/位": np.mean([250.0, 500.0]), 
    "露天16元/小时，室内700/月": 700.0, 
    "临保2.5元/小时月保400元/月": 400.0, 
    "临保2.5元/时/位，月保400元/月/位": 400.0, 
    "临保:4元/小时/位,月保:550元/位/月": 550.0, 
    "固定车位1400元/年，非固定车位100元/月": np.mean([1400.0 / 12.0, 100.0]), 
    "第一小时5块，后面1小时1块，一天15封顶": 15.0 * 30, 
    "地下400，地上免费": np.mean([400.0, 0.0]), 
    "地下350元/月/位 加60管理费/月": 350.0 + 60.0, 
    "地上免费  地下400元/月": np.mean([0.0, 400.0]), 
    "地上4元/小时/位": 4.0 * 8.0 * 30.0, 
    "地上150元/月/位，地下2元/时/位，地下固定车位450元/月/位": np.mean([150.0, 2.0 * 8.0 * 30.0, 450.0])
}

# 2. 定义硬编码+通用逻辑的解析函数
def parse_parking_fee_hardcoded(text_data):
    if not isinstance(text_data, str):
        return np.nan
    
    text_original = text_data.strip()
    text_lower = text_original.lower()

    # 优先级1：硬编码查找
    if text_original in hardcoded_parking_fees:
        return hardcoded_parking_fees[text_original]

    # 优先级2：明确免费/0值（含“暂无”）
    zero_keywords = ['免费', '没有停车费', '不收费']
    if any(keyword in text_lower for keyword in zero_keywords) or text_lower in ['无', '暂无', '0']:
        return 0.0
        
    # 优先级3：明确未知
    unknown_keywords = ['unknown', '未知', '无法核实', '无法获知']
    if any(keyword in text_lower for keyword in unknown_keywords):
        return np.nan

    # 优先级4：提取数字并平均
    numbers = re.findall(r'(\d+\.?\d*)', text_original)
    numbers = [float(n) for n in numbers if n] 
    if len(numbers) > 0:
        return np.mean(numbers)
        
    # 优先级5：无法解析
    return np.nan

# 3. 生成停车费特征
X_train['Parking_Fee'] = X_train['停车费用'].apply(parse_parking_fee_hardcoded)
X_val['Parking_Fee'] = X_val['停车费用'].apply(parse_parking_fee_hardcoded)
X_test_price['Parking_Fee'] = X_test_price['停车费用'].apply(parse_parking_fee_hardcoded)
print("已成功创建 'Parking_Fee' (此时仍包含 NaN)。")

# 4. 验证训练集中位数
print("\n--- 'Parking_Fee' 填充前验证 ---")
parking_fee_non_nan = X_train['Parking_Fee'].dropna() 
median_parking_fee = parking_fee_non_nan.median() 
num_non_nan = len(parking_fee_non_nan)
num_zeros = (parking_fee_non_nan == 0).sum()

print(f"总共 {num_non_nan} 行有已知的停车费数据。")
print(f"其中，有 {num_zeros} 行的值为 0 (占比: {num_zeros / num_non_nan:.1%})。")
print(f"用来填充的中位数是: {median_parking_fee:.2f}") 

# 5. 用训练集中位数填充NaN
print(f"\n--- 步骤 3.7.2 ：为 'Parking_Fee' 填充 NaN ---")
X_train['Parking_Fee'] = X_train['Parking_Fee'].fillna(median_parking_fee)
X_val['Parking_Fee'] = X_val['Parking_Fee'].fillna(median_parking_fee)
X_test_price['Parking_Fee'] = X_test_price['Parking_Fee'].fillna(median_parking_fee)
print(f"已使用中位数 {median_parking_fee:.2f} 填充 'Parking_Fee' 的 NaN 值。")

# 6. 验证硬编码结果
check_cols = ['停车费用', 'Parking_Fee']
print("\n--- 复杂样本检查 (硬编码验证) ---")
hardcoded_check_dict = hardcoded_parking_fees.copy()
hardcoded_check_dict["临保无法核实月保无法核实"] = median_parking_fee 
hardcoded_check_dict["暂无"] = 0.0

found_count = 0
fail_count = 0
failed_texts = []
for text, expected_value in hardcoded_check_dict.items():
    rows = X_train[X_train['停车费用'] == text]
    if not rows.empty:
        actual_value = rows['Parking_Fee'].iloc[0]
        current_expected = median_parking_fee if text == "临保无法核实月保无法核实" else expected_value
        
        if not np.isclose(actual_value, current_expected, atol=0.01): 
            fail_count += 1
            failed_texts.append(f"Text: '{text}', 期望: {current_expected:.2f}, 实际: {actual_value:.2f}")
        else:
            found_count += 1
            
print(f"成功找到并验证了 {found_count} 个指定的硬编码/特殊样本。")
if fail_count > 0:
    print(f"发现 {fail_count} 个检查失败:")
    for failure in failed_texts:
        print(f"  - {failure}")
else:
    print("所有硬编码/特殊样本检查均通过！")

# 7. 随机抽样验证
print("\n--- 随机样本 (5个) ---")
print(X_train[check_cols].sample(5, random_state=123))



--- 步骤 3.7.1 (硬编码特殊情况版)：开始高级量化 '停车费用' ---
直接映射指定特殊情况到期望值。
已成功创建 'Parking_Fee' (此时仍包含 NaN)。

--- 'Parking_Fee' 填充前验证 ---
总共 54686 行有已知的停车费数据。
其中，有 2472 行的值为 0 (占比: 4.5%)。
用来填充的中位数是: 300.00

--- 步骤 3.7.2 ：为 'Parking_Fee' 填充 NaN ---
已使用中位数 300.00 填充 'Parking_Fee' 的 NaN 值。

--- 复杂样本检查 (硬编码验证) ---
成功找到并验证了 20 个指定的硬编码/特殊样本。
所有硬编码/特殊样本检查均通过！

--- 随机样本 (5个) ---
          停车费用  Parking_Fee
57080  Unknown        300.0
81727      200        200.0
67143  Unknown        300.0
88318      150        150.0
84420  Unknown        300.0


In [33]:
print("--- 步骤 3.8 ：创建 Polynomial Features ---")

# --- 1. 创建 '建筑面积' 的平方项 ---
X_train['Area_sq'] = X_train['建筑面积'] ** 2
X_val['Area_sq'] = X_val['建筑面积'] ** 2
X_test_price['Area_sq'] = X_test_price['建筑面积'] ** 2

print("已成功创建 'Area_sq' (建筑面积的平方) 特征。")

# --- 2. 检查结果 (可选) ---
check_cols = ['建筑面积', 'Area_sq']
print("\n--- 新的 Area_sq 特征 (前5行) ---")
print(X_train[check_cols].head())

# --- 3. 更新维度 ---
print(f"\nX_train 新维度: {X_train.shape}") 
# (列数应该增加了 1)

--- 步骤 3.8 ：创建 Polynomial Features ---
已成功创建 'Area_sq' (建筑面积的平方) 特征。

--- 新的 Area_sq 特征 (前5行) ---
         建筑面积     Area_sq
20486   58.99   3479.8201
53134  143.13  20486.1969
8277    60.80   3696.6400
45156   71.95   5176.8025
99942   88.00   7744.0000

X_train 新维度: (83096, 70)


In [34]:
print("--- 步骤 3.9：开始处理分类特征 ---")

# --- 设置中文字体，避免警告 ---
import matplotlib.pyplot as plt
import matplotlib
import numpy as np  # 补充必要导入

# 简化字体设置逻辑
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("中文字体设置完成（若中文显示中文需手动系统无对应对应字体）")


# --- 1. 装修情况（多类别分类变量） ---
print("\n--- 处理 '装修情况' ---")

# 查看分布并创建独热编码
print("装修情况分布:")
print(X_train['装修情况'].value_counts())

# 生成虚拟变量并统一列（用reindex简化列对齐）
prefix = 'Renovation'
renovation_dummies = {
    'train': pd.get_dummies(X_train['装修情况'], prefix=prefix),
    'val': pd.get_dummies(X_val['装修情况'], prefix=prefix),
    'test': pd.get_dummies(X_test_price['装修情况'], prefix=prefix)
}

# 统一所有数据集的虚拟变量列（取并集后重索引）
all_cols = renovation_dummies['train'].columns.union(
    renovation_dummies['val'].columns.union(renovation_dummies['test'].columns)
)
for key in renovation_dummies:
    renovation_dummies[key] = renovation_dummies[key].reindex(columns=all_cols, fill_value=0)

# 合并回主数据集
X_train = pd.concat([X_train, renovation_dummies['train']], axis=1)
X_val = pd.concat([X_val, renovation_dummies['val']], axis=1)
X_test_price = pd.concat([X_test_price, renovation_dummies['test']], axis=1)

print(f"已创建 {len(all_cols)} 个装修情况虚拟变量")


# --- 2. 房屋年限（有序分类变量） ---
print("\n--- 处理 '房屋年限' ---")

# 查看分布并编码
print("房屋年限分布:")
print(X_train['房屋年限'].value_counts())

# 序数编码映射
house_age_mapping = {'Unknown': np.nan, '未满两年': 0, '满两年': 1, '满五年': 2}
for df in [X_train, X_val, X_test_price]:
    df['House_Age_Years_Encoded'] = df['房屋年限'].map(house_age_mapping)

# 中位数填充（复用训练集中位数）
median_house_age = X_train['House_Age_Years_Encoded'].median()
for df in [X_train, X_val, X_test_price]:
    df['House_Age_Years_Encoded'] = df['House_Age_Years_Encoded'].fillna(median_house_age)

print(f"已创建 'House_Age_Years_Encoded'，中位数填充值: {median_house_age}")

# 二进制特征（满五年/未满两年）
for df in [X_train, X_val, X_test_price]:
    df['Is_Over_5_Years'] = (df['House_Age_Years_Encoded'] == 2).astype(int)
    df['Is_Under_2_Years'] = (df['House_Age_Years_Encoded'] == 0).astype(int)

print("已创建 'Is_Over_5_Years' 和 'Is_Under_2_Years' 二进制特征")


# --- 3. 产权所属（二分类变量） ---
print("\n--- 处理 '产权所属' ---")

# 查看分布并编码
print("产权所属分布:")
print(X_train['产权所属'].value_counts())

# 二进制编码及众数填充
ownership_mapping = {'非共有': 0, '共有': 1, 'Unknown': np.nan}
for df in [X_train, X_val, X_test_price]:
    df['Ownership_Encoded'] = df['产权所属'].map(ownership_mapping)

mode_ownership = X_train['Ownership_Encoded'].mode()[0]  # 取众数
for df in [X_train, X_val, X_test_price]:
    df['Ownership_Encoded'] = df['Ownership_Encoded'].fillna(mode_ownership)

print(f"已创建 'Ownership_Encoded'，众数填充值: {mode_ownership}")


# --- 4. 创建交互特征 ---
print("\n--- 创建交互特征 ---")

# 装修*房屋年限交互（直接引用已对齐的虚拟变量列）
for df, dummies in zip(
    [X_train, X_val, X_test_price],
    [renovation_dummies['train'], renovation_dummies['val'], renovation_dummies['test']]
):
    df['Renovation_精装_Over_5_Years'] = dummies['Renovation_精装'] * df['Is_Over_5_Years']
    df['Ownership_毛坯_Interaction'] = df['Ownership_Encoded'] * dummies['Renovation_毛坯']

print("已创建 2 个交互特征")


# --- 5. 验证新特征 ---
print("\n--- 新创建的特征验证 ---")
new_categorical_features = [
    'House_Age_Years_Encoded', 'Is_Over_5_Years', 'Is_Under_2_Years',
    'Ownership_Encoded', 'Renovation_精装_Over_5_Years', 'Ownership_毛坯_Interaction'
]
renovation_cols = [col for col in X_train.columns if col.startswith('Renovation_')]

print("新特征统计:")
print(X_train[new_categorical_features].describe())
print(f"\n装修虚拟变量: {renovation_cols}")
print("虚拟变量前5行:")
print(X_train[renovation_cols].head())

print(f"\n数据集维度 - 训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test_price.shape}")


# --- 6. 特征选择准备 ---
print("\n--- 特征选择说明 ---")
print("后续建模将通过相关性分析、VIF、Lasso、树模型特征重要性进行特征选择")

# 相关性分析（简化标签转换逻辑）
try:
    temp_df = X_train[new_categorical_features + renovation_cols].copy()
    temp_df['Price'] = y_train.values
    corr = temp_df.corr()['Price'].sort_values(ascending=False).drop('Price')
    
    print("\n新特征与Price相关性（前10）:")
    print(corr.head(10))
    
    # 可视化（用字典映射英文标签）
    label_map = {
        'Renovation_精装': 'Renovation_Well',
        'Renovation_简装': 'Renovation_Simple',
        'Renovation_毛坯': 'Renovation_Raw',
        'Renovation_其他': 'Renovation_Other',
        'House_Age_Years_Encoded': 'House_Age_Years',
        'Is_Over_5_Years': 'Over_5_Years',
        'Is_Under_2_Years': 'Under_2_Years',
        'Ownership_Encoded': 'Ownership',
        'Renovation_精装_Over_5_Years': 'WellRenov_Over5Yrs',
        'Ownership_毛坯_Interaction': 'Ownership_Raw_Int'
    }
    top_features = corr.head(10)
    english_labels = [label_map.get(f, f) for f in top_features.index]
    
    plt.figure(figsize=(12, 6))
    plt.bar(english_labels, top_features.values)
    plt.xticks(rotation=45, ha='right')
    plt.title('Correlation with Price (Top 10 Features)')
    plt.ylabel('Correlation Coefficient')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"相关性分析暂不可用: {e}")

print("\n=== 分类特征工程完成 ===")

--- 步骤 3.9：开始处理分类特征 ---
中文字体设置完成（若中文显示中文需手动系统无对应对应字体）

--- 处理 '装修情况' ---
装修情况分布:
精装         37243
其他         19031
简装         16496
毛坯          9855
Unknown      471
Name: 装修情况, dtype: int64
已创建 5 个装修情况虚拟变量

--- 处理 '房屋年限' ---
房屋年限分布:
Unknown    35616
满五年        27934
满两年        12304
未满两年        7242
Name: 房屋年限, dtype: int64
已创建 'House_Age_Years_Encoded'，中位数填充值: 2.0
已创建 'Is_Over_5_Years' 和 'Is_Under_2_Years' 二进制特征

--- 处理 '产权所属' ---
产权所属分布:
非共有    59013
共有     24083
Name: 产权所属, dtype: int64
已创建 'Ownership_Encoded'，众数填充值: 0.0

--- 创建交互特征 ---
已创建 2 个交互特征

--- 新创建的特征验证 ---
新特征统计:
       House_Age_Years_Encoded  Is_Over_5_Years  Is_Under_2_Years  \
count             83096.000000     83096.000000      83096.000000   
mean                  1.677626         0.764778          0.087152   
std                   0.626704         0.424140          0.282060   
min                   0.000000         0.000000          0.000000   
25%                   2.000000         1.000000          0.000000   
50

<Figure size 864x432 with 1 Axes>


=== 分类特征工程完成 ===


In [35]:
import matplotlib.pyplot as plt
import warnings
import numpy as np
import seaborn as sns
import pandas as pd

# 设置避免警告
warnings.filterwarnings("ignore", category=FutureWarning)

# 设置pandas选项，避免use_inf_as_na的警告
pd.set_option('mode.use_inf_as_na', True)

print("--- 步骤 3.10 ：开始对目标变量 y (Price) 进行 Log Transformation ---")

# --- 1. 应用 np.log1p() ---
# 我们直接在 y_train 和 y_val 上操作
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print("成功对 y_train 和 y_val 应用 log1p 转换。")

# --- 2. (可选但推荐) 可视化对比 ---
# 我们可以画个图看看转换的效果

plt.figure(figsize=(12, 5))

# 图 1: 原始 Price 的分布（修正：将kde线条颜色参数移至line_kws）
plt.subplot(1, 2, 1)
train_clean = y_train.replace([np.inf, -np.inf], np.nan).dropna()
sns.histplot(
    train_clean, 
    kde=True,
    color="#3498db",  # 原始房价直方图用蓝色
    edgecolor="white",  # 白色边框避免柱子粘连
    kde_kws={},  # kde_kws不接受color参数，仅传递KDE估计器参数（如bw_adjust）
    line_kws={"color": "#2980b9", "linewidth": 2}  # 密度曲线颜色和线宽移至line_kws
)
sns.rugplot(train_clean, color="#3498db", alpha=0.5)  # 叠加数据点标注
plt.title('Original Price Distribution (y_train)')
plt.xlabel('Price')

# 图 2: Log(1+Price) 的分布（修正：将kde线条颜色参数移至line_kws）
plt.subplot(1, 2, 2)
train_log_clean = y_train_log.replace([np.inf, -np.inf], np.nan).dropna()
sns.histplot(
    train_log_clean, 
    kde=True,
    color="#2ecc71",  # 转换后对数房价用绿色
    edgecolor="white",  # 白色边框避免柱子粘连
    kde_kws={},  # 移除kde_kws中的color参数
    line_kws={"color": "#27ae60", "linewidth": 2}  # 密度曲线颜色和线宽移至line_kws
)
sns.rugplot(train_log_clean, color="#2ecc71", alpha=0.5)  # 叠加数据点标注
plt.title('Log(1+Price) Distribution (y_train_log)')
plt.xlabel('Log(1+Price)')

plt.tight_layout()  # 调整布局防止重叠
plt.show()

print("\n请观察上面两个图：第二个图（Log转换后）更加服从正态分布。")

# --- 3. 更新我们的目标变量 ---
print("\n现在目标变量是 y_train_log 和 y_val_log。")
print("最后评估时应使用 np.expm1() 将预测结果转换回来")

--- 步骤 3.10 ：开始对目标变量 y (Price) 进行 Log Transformation ---
成功对 y_train 和 y_val 应用 log1p 转换。


<Figure size 864x360 with 2 Axes>


请观察上面两个图：第二个图（Log转换后）更加服从正态分布。

现在目标变量是 y_train_log 和 y_val_log。
最后评估时应使用 np.expm1() 将预测结果转换回来


In [36]:
# 1. 统一导入模块（含异常处理依赖）
import pandas as pd
import numpy as np
import re
import sys  # 用于异常退出，避免后续连锁报错
from sklearn.model_selection import train_test_split


# 2. 配置化参数（集中管理，后续调参无需修改业务逻辑）
CONFIG = {
    "smoothing_factor": 20,  # 全局平滑因子（统一管理，改一处生效）
    "target_col_default": "Price",  # 目标列默认名
    "original_id_cols": ["城市", "区域", "板块"],  # 待删除的原始类别列
    "cols_to_drop_groups": {  # 待删除列分组（逻辑更清晰）
        "quantified1": ["交通出行", "周边配套", "核心卖点"],
        "quantified2": ["建筑年代", "房屋总数", "物 业 费", "交易时间", "上次交易"],
        "useless": ["开发商", "物业公司"],
        "advanced_quantified": ["房屋户型", "所在楼层", "梯户比例", "绿 化 率", "燃气费", "停车费用", "楼栋总数"],
        "encoded_categorical": ["装修情况", "房屋年限", "产权所属"]  # 新增：三个已编码的原始类别列
    }
}


# 3. 封装重复工具函数（减少代码冗余，降低维护成本）
def get_train_data_for_encoding(X_train: pd.DataFrame, y_train_log: pd.Series) -> pd.DataFrame:
    """
    封装：合并训练集特征与对数目标变量，确保索引一致（避免统计错误）
    Args:
        X_train: 预处理后的训练集特征
        y_train_log: 对数转换后的训练集目标变量
    Returns:
        合并后的训练集（含特征+目标变量）
    """
    # 校验索引一致性（Early Fail，提前发现问题）
    if not X_train.index.equals(y_train_log.index):
        raise ValueError("❌ X_train_pre_TE 与 y_train_log 索引不一致，会导致统计偏差！")
    
    # 合并数据
    train_data = pd.concat([X_train, y_train_log], axis=1)
    print(f"✅ 成功合并训练集特征（{X_train.shape[1]}列）与目标变量（1列）")
    return train_data


def apply_encoding_to_datasets(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    encode_col: str,  # 待编码的原始列（如"城市"）
    encode_map: dict,  # 编码映射字典
    fill_value: float,  # 缺失值填充值（全局均值）
    new_col_name: str  # 编码后的新列名（如"TE_City"）
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    封装：将编码映射应用到训练/验证/测试集，统一处理缺失值（避免重复代码）
    Returns:
        处理后的三个数据集（含新编码列）
    """
    # 训练集：直接映射（理论无缺失）
    X_train[new_col_name] = X_train[encode_col].map(encode_map)
    # 验证集/测试集：映射后用填充值补缺失（新类别）
    X_val[new_col_name] = X_val[encode_col].map(encode_map).fillna(fill_value)
    X_test[new_col_name] = X_test[encode_col].map(encode_map).fillna(fill_value)
    
    # 打印校验信息
    print(f"  - 训练集{new_col_name}缺失值：{X_train[new_col_name].isnull().sum()}")
    print(f"  - 验证集{new_col_name}缺失值：{X_val[new_col_name].isnull().sum()}")
    print(f"  - 测试集{new_col_name}缺失值：{X_test[new_col_name].isnull().sum()}")
    
    return X_train, X_val, X_test


# ---------------------- 4. 主流程：目标编码（保持原功能） ----------------------
print("="*50)
print("--- 目标编码准备阶段：数据清理 + 目标变量处理 ---")
print("="*50)

# 4.1 数据预处理：删除无用类别列（基于配置分组）
# 合并所有待删除列
all_cols_to_drop = []
for col_group in CONFIG["cols_to_drop_groups"].values():
    all_cols_to_drop.extend(col_group)
# 去重 + 只删除存在的列（避免KeyError）
all_cols_to_drop = list(set(all_cols_to_drop))  # 去重（防止分组重复）
cols_to_drop_final = [col for col in all_cols_to_drop if col in X_train.columns]

# 执行删除（避免inplace，保持数据不可变）
X_train_pre_TE = X_train.drop(columns=cols_to_drop_final, errors="ignore").copy()
X_val_pre_TE = X_val.drop(columns=cols_to_drop_final, errors="ignore").copy()
# 统一测试集命名：与后续步骤保持一致（原X_test_price → X_test_pre_TE）
X_test_pre_TE = X_test_price.drop(columns=cols_to_drop_final, errors="ignore").copy()

# 打印清理结果（更详细，便于调试）
print(f"✅ 数据清理完成：删除{len(cols_to_drop_final)}个无用列（{cols_to_drop_final}）")
print(f"   清理后维度（TE前）：训练集{X_train_pre_TE.shape} | 验证集{X_val_pre_TE.shape} | 测试集{X_test_pre_TE.shape}")
if X_train_pre_TE.shape[1] < 50:  # 粗略校验，避免误删关键列
    print("⚠️  警告：训练集列数过少（<50），请检查是否误删关键特征！")


# 4.2 目标变量处理：对数转换（防负数值）
print("\n--- 目标变量对数转换（log1p = log(1+x)，避免log(0)错误） ---")
# 校验原始目标变量无负值（Early Fail）
if (y_train < 0).any():
    raise ValueError("❌ 目标变量y_train存在负值，无法进行log1p转换！请检查原始数据。")
if (y_val < 0).any():
    raise ValueError("❌ 目标变量y_val存在负值，无法进行log1p转换！请检查原始数据。")

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
print(f"✅ 对数转换完成：原始y_train均值{y_train.mean():.2f} → 对数后均值{y_train_log.mean():.4f}")


# 4.3 计算全局均值（用于平滑）
global_mean_log_price = y_train_log.mean()
print(f"\n✅ 全局平均对数价格（平滑基准）：{global_mean_log_price:.6f}")
print("\n--- 准备就绪，开始目标编码 ---")


# ---------------------- 5. 分维度目标编码（城市→区域→板块） ----------------------
# 5.1 城市编码（TE_City）
print("\n" + "-"*40)
print("--- 步骤1：城市目标编码 + 平滑 ---")
print("-"*40)

# 合并训练集数据
train_data = get_train_data_for_encoding(X_train_pre_TE, y_train_log)
target_col_name = y_train_log.name if y_train_log.name is not None else CONFIG["target_col_default"]

# 计算城市统计信息（均值+样本量）
city_stats = train_data.groupby("城市")[target_col_name].agg(["mean", "count"]).reset_index()
print(f"✅ 从训练集计算{len(city_stats)}个城市的统计信息（示例前3个）：")
print(city_stats.head(3).to_string(index=False))  # 打印示例，直观确认

# 计算平滑编码值（使用配置的平滑因子）
city_stats["TE_City_Smoothed"] = (
    city_stats["count"] * city_stats["mean"] + CONFIG["smoothing_factor"] * global_mean_log_price
) / (city_stats["count"] + CONFIG["smoothing_factor"])

# 生成编码映射字典
city_encoding_map = dict(zip(city_stats["城市"], city_stats["TE_City_Smoothed"]))
print(f"✅ 生成{len(city_encoding_map)}个城市的编码映射")

# 应用编码到三个数据集（调用封装函数）
X_train_pre_TE, X_val_pre_TE, X_test_pre_TE = apply_encoding_to_datasets(
    X_train=X_train_pre_TE,
    X_val=X_val_pre_TE,
    X_test=X_test_pre_TE,
    encode_col="城市",
    encode_map=city_encoding_map,
    fill_value=global_mean_log_price,
    new_col_name="TE_City"
)


# 5.2 区域编码（TE_District）
print("\n" + "-"*40)
print("--- 步骤2：区域目标编码 + 平滑 ---")
print("-"*40)

# 合并训练集（复用封装函数，无需重复try-except）
train_data = get_train_data_for_encoding(X_train_pre_TE, y_train_log)

# 计算区域统计信息
district_stats = train_data.groupby("区域")[target_col_name].agg(["mean", "count"]).reset_index()
print(f"✅ 从训练集计算{len(district_stats)}个区域的统计信息")

# 计算平滑编码值
district_stats["TE_District_Smoothed"] = (
    district_stats["count"] * district_stats["mean"] + CONFIG["smoothing_factor"] * global_mean_log_price
) / (district_stats["count"] + CONFIG["smoothing_factor"])

# 生成编码映射
district_encoding_map = dict(zip(district_stats["区域"], district_stats["TE_District_Smoothed"]))
print(f"✅ 生成{len(district_encoding_map)}个区域的编码映射")

# 应用编码（调用封装函数）
X_train_pre_TE, X_val_pre_TE, X_test_pre_TE = apply_encoding_to_datasets(
    X_train=X_train_pre_TE,
    X_val=X_val_pre_TE,
    X_test=X_test_pre_TE,
    encode_col="区域",
    encode_map=district_encoding_map,
    fill_value=global_mean_log_price,
    new_col_name="TE_District"
)


# 5.3 板块编码（TE_Block_Stratified，分层平滑）
print("\n" + "-"*40)
print("--- 步骤3：板块目标编码 + 分层平滑 ---")
print("-"*40)

# 合并训练集
train_data = get_train_data_for_encoding(X_train_pre_TE, y_train_log)

# 计算板块统计信息（按"区域+板块"分组，关联父级区域）
block_stats = train_data.groupby(["区域", "板块"])[target_col_name].agg(["mean", "count"])
print(f"✅ 从训练集计算{len(block_stats)}个板块的统计信息（MultiIndex分组）")

# 获取区域原始均值（用于分层平滑，校验变量存在）
try:
    district_mean_map = dict(zip(district_stats["区域"], district_stats["mean"]))  # 用区域原始均值（未平滑）
except NameError:
    print("❌ 错误：district_stats未定义！请先运行「区域编码步骤」再执行板块编码。")
    sys.exit(1)  # 主动退出，避免后续报错


# 定义分层平滑函数（增强注释，说明逻辑）
def stratified_smooth(row: pd.Series) -> float:
    """
    板块分层平滑：用「父级区域均值」替代全局均值作为平滑基准，更贴合地理关联性
    Args:
        row: block_stats的单行数据（含mean/count，索引为(区域,板块)）
    Returns:
        平滑后的板块编码值
    """
    block_mean = row["mean"]  # 板块自身的对数房价均值
    block_count = row["count"]  # 板块样本量
    district_id = row.name[0]  # 从MultiIndex提取父级区域名
    # 父级均值：优先用区域原始均值，无则用全局均值（降级策略）
    parent_mean = district_mean_map.get(district_id, global_mean_log_price)
    
    # 分层平滑公式
    smoothed_val = (block_count * block_mean + CONFIG["smoothing_factor"] * parent_mean) / (block_count + CONFIG["smoothing_factor"])
    return smoothed_val


# 计算板块平滑编码值
block_stats["TE_Block_Stratified"] = block_stats.apply(stratified_smooth, axis=1)
# 生成板块编码映射（键为(区域,板块)元组，避免同名板块混淆）
block_encoding_map = block_stats["TE_Block_Stratified"].to_dict()
print(f"✅ 生成{len(block_encoding_map)}个板块的分层平滑编码映射")


# 应用板块编码（三级回退逻辑，增强注释）
def apply_block_encoding(row: pd.Series) -> float:
    """
    板块编码应用：三级回退（确保无缺失值）
    1. 优先用(区域,板块)精准匹配 → 2. 回退到区域编码值 → 3. 回退到全局均值
    """
    key = (row["区域"], row["板块"])
    # 回退1：精准匹配板块
    if key in block_encoding_map:
        return block_encoding_map[key]
    # 回退2：匹配区域
    elif row["区域"] in district_encoding_map:
        return district_encoding_map[row["区域"]]
    # 回退3：全局均值兜底
    else:
        return global_mean_log_price


# 应用到三个数据集
X_train_pre_TE["TE_Block_Stratified"] = X_train_pre_TE.apply(apply_block_encoding, axis=1)
X_val_pre_TE["TE_Block_Stratified"] = X_val_pre_TE.apply(apply_block_encoding, axis=1)
X_test_pre_TE["TE_Block_Stratified"] = X_test_pre_TE.apply(apply_block_encoding, axis=1)

# 校验缺失值
print(f"✅ 板块编码应用完成：")
print(f"  - 训练集TE_Block_Stratified缺失值：{X_train_pre_TE['TE_Block_Stratified'].isnull().sum()}")
print(f"  - 验证集TE_Block_Stratified缺失值：{X_val_pre_TE['TE_Block_Stratified'].isnull().sum()}")
print(f"  - 测试集TE_Block_Stratified缺失值：{X_test_pre_TE['TE_Block_Stratified'].isnull().sum()}")


# ---------------------- 6. 最终清理：删除原始ID列 ----------------------
print("\n" + "-"*40)
print("---  清理原始 ID 列 ('城市', '区域', '板块') ---")
print("-"*40)

# 只删除存在的列（避免KeyError）
cols_to_drop_id = [col for col in CONFIG["original_id_cols"] if col in X_train_pre_TE.columns]
try:
    if cols_to_drop_id:
        X_train_post_TE = X_train_pre_TE.drop(columns=cols_to_drop_id).copy()
        X_val_post_TE = X_val_pre_TE.drop(columns=cols_to_drop_id).copy()
        X_test_post_TE = X_test_pre_TE.drop(columns=cols_to_drop_id).copy()  # 统一测试集命名
        print(f"✅ 已从数据集中移除原始 ID 列: {cols_to_drop_id}")
    else:
        X_train_post_TE = X_train_pre_TE.copy()  # 如果列不存在，保持不变
        X_val_post_TE = X_val_pre_TE.copy()
        X_test_post_TE = X_test_pre_TE.copy()
        print("ℹ️  原始 ID 列在数据集中未找到，可能已被删除。")

    # 检查最终维度（保留步骤6.8的预期列数注释）
    print(f"\n清理原始 ID 列后的维度:")
    print(f"X_train: {X_train_post_TE.shape}")  # 列数应为 53 + 3 (TE features) - 3 (Original IDs) = 53
    print(f"X_val:   {X_val_post_TE.shape}")
    print(f"X_test:  {X_test_post_TE.shape}")

except NameError:
    print("❌ 错误：无法找到 X_train_pre_TE。请确保之前的步骤已成功运行。")


print("\n" + "="*50)
print("--- 目标编码全流程完成！下一步：对剩余类别特征进行One-Hot编码 ---")
print("="*50)

--- 目标编码准备阶段：数据清理 + 目标变量处理 ---
✅ 数据清理完成：删除20个无用列（['交易时间', '产权所属', '物 业 费', '上次交易', '楼栋总数', '停车费用', '所在楼层', '物业公司', '建筑年代', '交通出行', '开发商', '周边配套', '装修情况', '房屋户型', '绿 化 率', '房屋总数', '梯户比例', '房屋年限', '燃气费', '核心卖点']）
   清理后维度（TE前）：训练集(83096, 61) | 验证集(20775, 61) | 测试集(34017, 61)

--- 目标变量对数转换（log1p = log(1+x)，避免log(0)错误） ---
✅ 对数转换完成：原始y_train均值2269478.97 → 对数后均值14.2646

✅ 全局平均对数价格（平滑基准）：14.264576

--- 准备就绪，开始目标编码 ---

----------------------------------------
--- 步骤1：城市目标编码 + 平滑 ---
----------------------------------------
✅ 成功合并训练集特征（61列）与目标变量（1列）
✅ 从训练集计算12个城市的统计信息（示例前3个）：
 城市      mean  count
  0 15.118860  13269
  1 13.739871   5197
  2 13.772165  19910
✅ 生成12个城市的编码映射
  - 训练集TE_City缺失值：0
  - 验证集TE_City缺失值：0
  - 测试集TE_City缺失值：0

----------------------------------------
--- 步骤2：区域目标编码 + 平滑 ---
----------------------------------------
✅ 成功合并训练集特征（62列）与目标变量（1列）
✅ 从训练集计算115个区域的统计信息
✅ 生成115个区域的编码映射
  - 训练集TE_District缺失值：0
  - 验证集TE_District缺失值：0
  - 测试集TE_District缺失值：0

-----------------------

In [37]:
# 整合特征工程收尾流程：OHE编码+对齐 → 最终清理 → Winsorization处理
import pandas as pd
import numpy as np

# ---------------------- 步骤3.13：最终OHE编码+对齐+重复列处理 ----------------------
print("\n--- 步骤3.13(修改drop_first)：开始最终OHE (drop_first=True)与对齐 ---")

try:
    if 'X_train_post_TE' not in locals():
        raise NameError("X_train_post_TE not defined")
    
    # 识别剩余需编码的类别特征
    categorical_cols_remaining_for_ohe = X_train_post_TE.select_dtypes(
        include=['object', 'category', 'string']
    ).columns.tolist()
    print(f"将对 {len(categorical_cols_remaining_for_ohe)} 个列进行OHE (drop_first=True)...")

    if not categorical_cols_remaining_for_ohe:
        print("警告：未找到需要OHE的列。")
        X_train_final = X_train_post_TE.copy()
        X_val_final = X_val_post_TE.copy()
        X_test_final = X_test_post_TE.copy()  # 修改：去掉price后缀
    else:
        print("正在执行最终One-Hot编码 (drop_first=True)...")
        # 执行OHE编码（测试集输入改为X_test_post_TE，输出改为X_test_final）
        X_train_final = pd.get_dummies(
            X_train_post_TE, 
            columns=categorical_cols_remaining_for_ohe, 
            dummy_na=False, 
            dtype=int, 
            drop_first=True
        )
        X_val_final = pd.get_dummies(
            X_val_post_TE, 
            columns=categorical_cols_remaining_for_ohe, 
            dummy_na=False, 
            dtype=int, 
            drop_first=True
        )
        X_test_final = pd.get_dummies(  # 修改：去掉price后缀
            X_test_post_TE,  # 输入改为X_test_post_TE（与前面代码匹配）
            columns=categorical_cols_remaining_for_ohe, 
            dummy_na=False, 
            dtype=int, 
            drop_first=True
        )
        print("编码完成。")

        # 定义重复列删除函数
        def remove_duplicate_cols(df, df_name):
            duplicate_cols = df.columns[df.columns.duplicated()].tolist()
            if duplicate_cols:
                print(f"警告：{df_name}存在{len(duplicate_cols)}个重复列，已删除重复项（保留第一个）：")
                print(f"重复列名：{duplicate_cols}")
                df = df.loc[:, ~df.columns.duplicated(keep='first')]
            else:
                print(f"{df_name}无重复列，无需处理。")
            return df

        # 删除重复列（测试集变量名改为X_test_final）
        X_train_final = remove_duplicate_cols(X_train_final, "X_train_final")
        X_val_final = remove_duplicate_cols(X_val_final, "X_val_final")
        X_test_final = remove_duplicate_cols(X_test_final, "X_test_final")  # 修改：去掉price后缀

        # 数据集列对齐（测试集用X_test_final）
        print("正在执行最终对齐...")
        final_train_columns = X_train_final.columns
        X_val_final = X_val_final.reindex(columns=final_train_columns, fill_value=0)
        X_test_final = X_test_final.reindex(columns=final_train_columns, fill_value=0)  # 修改：去掉price后缀
        print("对齐完成。")

    # 输出编码后维度信息（测试集用X_test_final）
    print(f"\nOHE (drop_first=True)和对齐后维度:")
    print(f"X_train_final: {X_train_final.shape}")
    print(f"X_val_final:   {X_val_final.shape}")
    print(f"X_test_final: {X_test_final.shape}")  # 修改：去掉price后缀

except NameError as e:
    print(f"错误：{e}。请确保目标编码和ID清理步骤(Step 6.8)已成功运行。")
except Exception as e:
    print(f"发生意外错误: {e}")


# ---------------------- 步骤3.14：最终清理（删除冗余特征） ----------------------
print("\n--- 步骤3.14 (重新运行)：最终清理 ---")

# 定义需删除的冗余列
final_cols_to_drop = [
    '套内面积', '年份', '区县', '板块_comm',
    'coord_x', 'coord_y',
    'Total_Floors',
    'Num_Elevators', 'Num_Households'
]

try:
    if 'X_train_final' not in locals():
        raise NameError("X_train_final (OHE drop_first输出)未定义")
    
    # 仅删除存在的列（测试集用X_test_final）
    cols_present_train = [col for col in final_cols_to_drop if col in X_train_final.columns]
    if cols_present_train:
        X_train_final_cleaned = X_train_final.drop(columns=cols_present_train)
        X_val_final_cleaned = X_val_final.drop(columns=cols_present_train)
        X_test_final_cleaned = X_test_final.drop(columns=cols_present_train)  # 修改：去掉price后缀
        print(f"已创建_cleaned变量，移除了: {cols_present_train}")
    else:
        X_train_final_cleaned = X_train_final.copy()
        X_val_final_cleaned = X_val_final.copy()
        X_test_final_cleaned = X_test_final.copy()  # 修改：去掉price后缀
        print("需要移除的列未找到。")

    # 输出清理后维度信息（测试集用X_test_final_cleaned）
    print(f"\n最终清理后的维度:")
    final_num_features_check_dr = X_train_final_cleaned.shape[1]
    print(f"X_train_final_cleaned: ({X_train_final_cleaned.shape[0]}, {final_num_features_check_dr})")
    print(f"X_val_final_cleaned:   ({X_val_final_cleaned.shape[0]}, {final_num_features_check_dr})")
    print(f"X_test_final_cleaned:  ({X_test_final_cleaned.shape[0]}, {final_num_features_check_dr})")  # 修改：去掉price后缀

except NameError as e:
    print(f"错误：{e}。请确保上一步OHE(drop_first=True)已成功运行。")
except Exception as e:
    print(f"最终清理时发生意外错误: {e}")


# ---------------------- 步骤3.15：Winsorization（缩尾处理异常值） ----------------------
print("\n--- 步骤3.15(重新运行)：开始Winsorization ---")

# 定义需进行缩尾处理的特征
original_numeric_cols_to_keep = [
    '建筑面积', 'lon', 'lat', '容 积 率', '停车位', 
    'Has_交通出行', 'Has_周边配套', 'Has_核心卖点', 'House_Age', 
    'Property_Fee', 'Total_Units', 'Transaction_Year', 'Transaction_Month', 
    'Years_Since_Last_Transaction', 'Num_Rooms', 'Num_LivingRooms', 
    'Num_Kitchens', 'Num_Baths', 'Floor_Ratio', 'Estimated_Floor', 
    'Elevator_Ratio', 'Greening_Rate', 'Gas_Fee', 'Parking_Fee', 
    'Total_Buildings', 'Area_sq'
]
TE_cols = ['TE_City', 'TE_District', 'TE_Block_Stratified']
cols_to_winsorize_final = original_numeric_cols_to_keep + TE_cols

try:
    # 筛选数据中实际存在的列（测试集用X_test_final_cleaned）
    cols_to_winsorize_present = [
        col for col in cols_to_winsorize_final 
        if col in X_train_final_cleaned.columns
    ]
    print(f"将对 {len(cols_to_winsorize_present)} 个特征进行Winsorization...")

    # 从训练集学习缩尾界限
    lower_quantile = 0.01
    upper_quantile = 0.99
    bounds = {}
    print("正在从X_train_final_cleaned学习界限...")
    for col in cols_to_winsorize_present:
        q1 = X_train_final_cleaned[col].quantile(lower_quantile)
        q99 = X_train_final_cleaned[col].quantile(upper_quantile)
        if q1 < q99:
            bounds[col] = (q1, q99)

    # 应用缩尾处理（测试集用X_test_final_cleaned，输出改为X_test_winsorized）
    print("正在应用界限...")
    X_train_winsorized = X_train_final_cleaned.copy()
    X_val_winsorized = X_val_final_cleaned.copy()
    X_test_winsorized = X_test_final_cleaned.copy()  # 修改：去掉price后缀

    for col in bounds:
        q1, q99 = bounds[col]
        X_train_winsorized[col] = X_train_winsorized[col].clip(lower=q1, upper=q99)
        X_val_winsorized[col] = X_val_winsorized[col].clip(lower=q1, upper=q99)
        X_test_winsorized[col] = X_test_winsorized[col].clip(lower=q1, upper=q99)  # 修改：去掉price后缀

    print("Winsorization完成。")

except Exception as e:
    print(f"Winsorization处理发生错误: {e}")


--- 步骤3.13(修改drop_first)：开始最终OHE (drop_first=True)与对齐 ---
将对 12 个列进行OHE (drop_first=True)...
正在执行最终One-Hot编码 (drop_first=True)...
编码完成。
X_train_final无重复列，无需处理。
X_val_final无重复列，无需处理。
X_test_final无重复列，无需处理。
正在执行最终对齐...
对齐完成。

OHE (drop_first=True)和对齐后维度:
X_train_final: (83096, 704)
X_val_final:   (20775, 704)
X_test_final: (34017, 704)

--- 步骤3.14 (重新运行)：最终清理 ---
已创建_cleaned变量，移除了: ['套内面积', '年份', '区县', '板块_comm', 'coord_x', 'coord_y', 'Total_Floors', 'Num_Elevators', 'Num_Households']

最终清理后的维度:
X_train_final_cleaned: (83096, 695)
X_val_final_cleaned:   (20775, 695)
X_test_final_cleaned:  (34017, 695)

--- 步骤3.15(重新运行)：开始Winsorization ---
将对 29 个特征进行Winsorization...
正在从X_train_final_cleaned学习界限...
正在应用界限...
Winsorization完成。


In [38]:
# 缩尾后的数据缩放（适配线性模型）
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# 检查缩尾后数据集是否存在
try:
    assert 'X_train_winsorized' in locals() and 'X_val_winsorized' in locals() and 'X_test_winsorized' in locals()
    print("✅ 缩尾后数据集存在，开始特征缩放...")
except AssertionError:
    raise ValueError("❌ 请先运行缩尾处理，确保 X_train_winsorized/X_val_winsorized/X_test_winsorized 存在")

# 初始化缩放器（仅用训练集拟合，防数据泄露）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_winsorized)
X_val_scaled = scaler.transform(X_val_winsorized)
X_test_scaled = scaler.transform(X_test_winsorized)

# 转换为DataFrame（保留列名和索引）
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_winsorized.columns, index=X_train_winsorized.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val_winsorized.columns, index=X_val_winsorized.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_winsorized.columns, index=X_test_winsorized.index)

# 验证缩放效果
print("\n--- 缩放效果验证 ---")
rand_col = X_train_scaled_df.columns[0]
print(f"特征 '{rand_col}' 缩放后：均值 {X_train_scaled_df[rand_col].mean():.4f}（≈0），方差 {X_train_scaled_df[rand_col].var():.4f}（≈1）")
print(f"数据集维度：训练集 {X_train_scaled_df.shape}，验证集 {X_val_scaled_df.shape}，测试集 {X_test_scaled_df.shape}")

# 保存缩放器
joblib.dump(scaler, 'linear_model_scaler.pkl')
print("\n✅ 缩放器已保存为 'linear_model_scaler.pkl'")

# 校验特征与目标变量索引一致性
try:
    assert X_train_scaled_df.index.equals(y_train_log.index) and X_val_scaled_df.index.equals(y_val_log.index)
    print("✅ 缩放后特征与目标变量索引一致，可用于建模")
except AssertionError:
    raise ValueError("❌ 特征与目标变量索引不匹配，请检查数据对齐")

print("\n--- 可用数据集 ---")
print(f"训练集：{X_train_scaled_df.shape}（X_train_scaled_df）")
print(f"验证集：{X_val_scaled_df.shape}（X_val_scaled_df）")
print(f"测试集：{X_test_scaled_df.shape}（X_test_scaled_df）")

✅ 缩尾后数据集存在，开始特征缩放...

--- 缩放效果验证 ---
特征 '建筑面积' 缩放后：均值 0.0000（≈0），方差 1.0000（≈1）
数据集维度：训练集 (83096, 695)，验证集 (20775, 695)，测试集 (34017, 695)

✅ 缩放器已保存为 'linear_model_scaler.pkl'
✅ 缩放后特征与目标变量索引一致，可用于建模

--- 可用数据集 ---
训练集：(83096, 695)（X_train_scaled_df）
验证集：(20775, 695)（X_val_scaled_df）
测试集：(34017, 695)（X_test_scaled_df）


In [42]:
# --- 步骤 4.1：开始训练 OLS (Linear Regression) 模型 ---

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd
import warnings


# --- 1. 评估函数 (包含溢出处理) ---
try:
    evaluate_model_safe
except NameError:
    print("重新定义 evaluate_model_safe 函数...")
    def evaluate_model_safe(y_true_log, y_pred_log, model_name="Model"):
        mae, rmse = np.nan, np.nan 
        y_true_orig = np.expm1(y_true_log)
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                y_pred_orig = np.expm1(y_pred_log)
                if not np.all(np.isfinite(y_pred_orig)):
                     raise ValueError("Overflow or NaN after expm1")
                mae = mean_absolute_error(y_true_orig, y_pred_orig)
                rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
            except (OverflowError, ValueError) as e:
                print(f"错误：计算 {model_name} 原始尺度指标时发生溢出或无效值！将返回 NaN。")
        print(f"--- {model_name} Performance ---")
        if not np.isnan(mae):
            print(f"MAE (原始价格): {mae:,.2f}") 
            print(f"RMSE (原始价格): {rmse:,.2f}")
        else:
            print(f"MAE (原始价格): N/A (计算失败)")
            print(f"RMSE (原始价格): N/A (计算失败)")
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
        print(f"MAE (对数尺度): {mae_log:.4f}") 
        print(f"RMSE (对数尺度): {rmse_log:.4f}")
        return mae, rmse


# --- 2. 初始化并训练 OLS 模型 ---
ols_model = LinearRegression()
print("\n正在训练 OLS 模型 ...")
# 确保数据已正确预处理
if 'X_train_final' not in globals() or 'y_train_log' not in globals():
    raise ValueError("请确保已定义 X_train_final 和 y_train_log（预处理后的训练数据）")
ols_model.fit(X_train_final, y_train_log) 
print("OLS 模型训练完毕。")


# --- 3. 样本内评估 ---
print("\n--- (A) 样本内 (In-sample) 评估 ---")
y_train_pred_log_ols = ols_model.predict(X_train_final)
mae_in_ols, rmse_in_ols = evaluate_model_safe(y_train_log, y_train_pred_log_ols, "OLS In-sample")
train_r2_ols = ols_model.score(X_train_final, y_train_log)
print(f"R² (训练集): {train_r2_ols:.4f}")


# --- 4. 样本外评估 ---
print("\n--- (B) 样本外 (Out-of-sample) 评估 ---")
if 'X_val_final' not in globals() or 'y_val_log' not in globals():
    raise ValueError("请确保已定义 X_val_final 和 y_val_log（预处理后的验证数据）")
y_val_pred_log_ols = ols_model.predict(X_val_final) 
mae_out_ols, rmse_out_ols = evaluate_model_safe(y_val_log, y_val_pred_log_ols, "OLS Out-of-sample")
val_r2_ols = ols_model.score(X_val_final, y_val_log)
print(f"R² (验证集): {val_r2_ols:.4f}")


# --- 5. 交叉验证评估 ---
print("\n--- (C) 交叉验证评估 ---")
if 'X_train_scaled_df' not in globals():
    raise ValueError("请确保已定义 X_train_scaled_df（缩放后的训练特征）")
# 检查交叉验证结果是否异常
cv_r2_scores = cross_val_score(
    ols_model, 
    X_train_scaled_df, 
    y_train_log, 
    cv=6, 
    scoring='r2'
)
cv_r2_ols = cv_r2_scores.mean()
print(f"6折交叉验证 R² 分数: {cv_r2_scores}")  # 打印各折分数，便于排查异常
print(f"6折交叉验证平均 R²: {cv_r2_ols:.4f}")

# --- 6. 记录结果（仅保留OLS列） ---
# 定义完整的性能指标索引（7个元素）
performance_index = ['MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'R²_in', 'R²_out', 'R²_cv']

try:
    # 若性能表已存在，先清除其他模型的列，只保留OLS相关
    # 只保留当前需要的OLS列，删除其他模型列
    model_performance = model_performance.reindex(performance_index)  # 确保索引正确
    model_performance = model_performance[['OLS']] if 'OLS' in model_performance.columns else model_performance
    # 更新OLS列
    model_performance['OLS'] = [
        mae_in_ols, rmse_in_ols,
        mae_out_ols, rmse_out_ols,
        train_r2_ols, val_r2_ols, cv_r2_ols
    ]
except NameError:
    # 若性能表不存在，创建并仅初始化OLS列
    print("\n创建 model_performance DataFrame...")
    model_performance = pd.DataFrame(
        index=performance_index,
        data={'OLS': [
            mae_in_ols, rmse_in_ols,
            mae_out_ols, rmse_out_ols,
            train_r2_ols, val_r2_ols, cv_r2_ols
        ]}
    )

print("\n--- OLS 模型性能已记录 ---")
print(model_performance)



正在训练 OLS 模型 ...
OLS 模型训练完毕。

--- (A) 样本内 (In-sample) 评估 ---
--- OLS In-sample Performance ---
MAE (原始价格): 413,023.91
RMSE (原始价格): 938,378.27
MAE (对数尺度): 0.1754
RMSE (对数尺度): 0.2353
R² (训练集): 0.9199

--- (B) 样本外 (Out-of-sample) 评估 ---
错误：计算 OLS Out-of-sample 原始尺度指标时发生溢出或无效值！将返回 NaN。
--- OLS Out-of-sample Performance ---
MAE (原始价格): N/A (计算失败)
RMSE (原始价格): N/A (计算失败)
MAE (对数尺度): 8.5332
RMSE (对数尺度): 491.7556
R² (验证集): -357073.7651

--- (C) 交叉验证评估 ---
6折交叉验证 R² 分数: [-2.06699041e+25 -3.46948712e+24 -5.44813650e+24 -1.45351272e+25
 -1.82445190e+25 -5.37066843e+24]
6折交叉验证平均 R²: -11289640389821545215688704.0000

--- OLS 模型性能已记录 ---
                   OLS
MAE_in    4.130239e+05
RMSE_in   9.383783e+05
MAE_out            NaN
RMSE_out           NaN
R²_in     9.199472e-01
R²_out   -3.570738e+05
R²_cv    -1.128964e+25


In [43]:
# --- 步骤 15.1 (最终执行)：LassoCV ---

from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score  # 新增：用于交叉验证R²计算
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd 
import warnings 
import time 

print("--- 步骤 15.1：使用 LassoCV 进行超参数调优 (最终执行) ---")

# --- 1. 定义 alpha 候选范围 ---
# (我们仍然从较小的值开始搜索)
alphas_lasso = np.logspace(-4, -1.5, 10) # 搜索范围 0.0001 到 ~0.03 
print(f"Lasso 将尝试以下 alpha 值: {alphas_lasso}")

# --- 2. 初始化并运行 LassoCV ---
lasso_cv_model = LassoCV(alphas=alphas_lasso, 
                         cv=6, # <-- 使用作业要求的 6 折
                         random_state=111, 
                         n_jobs=4, # <-- 保持较低的 n_jobs (4 或 8)
                         max_iter=5000, 
                         tol=1e-3, # <-- 保持较宽松的 tol
                         verbose=1) # <-- 保留进度指示

print(f"正在运行 LassoCV (cv=6, n_jobs=4)...")
start_time = time.time() 
# --- 使用最终的 drop_first=True, 清理后的数据 ---
lasso_cv_model.fit(X_train_scaled_df, y_train_log) 
end_time = time.time() 
elapsed_time = end_time - start_time
print(f"\nLassoCV 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳 alpha ---
best_alpha_lasso = lasso_cv_model.alpha_
print(f"\nLassoCV 找到的最佳 alpha 是: {best_alpha_lasso:.6f}")

# --- 4. 使用最佳 alpha 模型进行评估 ---
best_lasso_model = lasso_cv_model 
coeffs_best_lasso = best_lasso_model.coef_
num_zero_coeffs_best = np.sum(coeffs_best_lasso == 0)
num_non_zero_coeffs_best = np.sum(coeffs_best_lasso != 0)
print(f"\n使用最佳 alpha 的 Lasso 模型选择了 {num_non_zero_coeffs_best} 个特征 / {X_train_scaled_df.shape[1]}。")

print("\n--- (A) 样本内 (In-sample) 评估 (Best Lasso) ---")
y_train_pred_log_best_lasso = best_lasso_model.predict(X_train_scaled_df)
mae_in_best_lasso, rmse_in_best_lasso = evaluate_model_safe(y_train_log, y_train_pred_log_best_lasso, "Best Lasso In-sample")
# 新增：样本内R²计算
train_r2_lasso = best_lasso_model.score(X_train_scaled_df, y_train_log)
print(f"R² (训练集): {train_r2_lasso:.4f}")

print("\n--- (B) 样本外 (Out-of-sample) 评估 (Best Lasso) ---")
y_val_pred_log_best_lasso = best_lasso_model.predict(X_val_scaled_df) 
mae_out_best_lasso, rmse_out_best_lasso = evaluate_model_safe(y_val_log, y_val_pred_log_best_lasso, "Best Lasso Out-of-sample")
# 新增：样本外R²计算
val_r2_lasso = best_lasso_model.score(X_val_scaled_df, y_val_log)
print(f"R² (验证集): {val_r2_lasso:.4f}")

# --- 新增：交叉验证R²计算（6折） ---
print("\n--- (C) 交叉验证评估 (Best Lasso) ---")
cv_r2_lasso = cross_val_score(
    best_lasso_model, 
    X_train_scaled_df,  # 基于训练特征集进行交叉验证
    y_train_log, 
    cv=6,  # 与调优时一致的6折
    scoring='r2'  # 评估指标为R²
).mean()  # 取6折平均值
print(f"6折交叉验证平均 R²: {cv_r2_lasso:.4f}")

# --- 5. 更新性能记录表（包含R²指标） ---
try:
    # 移除之前的无效 Lasso(1.0) 列 (如果存在)
    if 'Lasso (alpha=1.0)' in model_performance.columns:
         model_performance = model_performance.drop(columns=['Lasso (alpha=1.0)'])
    # 添加新的最佳 Lasso 结果（包含MAE、RMSE、R²）
    model_performance[f'Lasso (alpha={best_alpha_lasso:.4f})'] = [
        mae_in_best_lasso, rmse_in_best_lasso,  # 样本内MAE、RMSE
        mae_out_best_lasso, rmse_out_best_lasso,  # 样本外MAE、RMSE
        train_r2_lasso, val_r2_lasso, cv_r2_lasso  # 新增的R²指标
    ]
except NameError:
    # 若model_performance未定义，创建并初始化（包含R²索引）
    print("\n警告：model_performance DataFrame 未定义，正在创建...")
    model_performance = pd.DataFrame(
        index=['MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'R²_in', 'R²_out', 'R²_cv']
    )
    model_performance[f'Lasso (alpha={best_alpha_lasso:.4f})'] = [
        mae_in_best_lasso, rmse_in_best_lasso,
        mae_out_best_lasso, rmse_out_best_lasso,
        train_r2_lasso, val_r2_lasso, cv_r2_lasso
    ]

print("\n--- Best Lasso 模型性能已记录 ---")
print(model_performance)

--- 步骤 15.1：使用 LassoCV 进行超参数调优 (最终执行) ---
Lasso 将尝试以下 alpha 值: [0.0001     0.00018957 0.00035938 0.00068129 0.00129155 0.00244844
 0.00464159 0.00879923 0.01668101 0.03162278]
正在运行 LassoCV (cv=6, n_jobs=4)...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
............................................................[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   14.9s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:   14.9s finished



LassoCV 完成。耗时: 165.32 秒

LassoCV 找到的最佳 alpha 是: 0.000100

使用最佳 alpha 的 Lasso 模型选择了 655 个特征 / 695。

--- (A) 样本内 (In-sample) 评估 (Best Lasso) ---
--- Best Lasso In-sample Performance ---
MAE (原始价格): 414,791.84
RMSE (原始价格): 948,672.54
MAE (对数尺度): 0.1768
RMSE (对数尺度): 0.2365
R² (训练集): 0.9192

--- (B) 样本外 (Out-of-sample) 评估 (Best Lasso) ---
--- Best Lasso Out-of-sample Performance ---
MAE (原始价格): 412,672.41
RMSE (原始价格): 951,565.63
MAE (对数尺度): 0.1771
RMSE (对数尺度): 0.2364
R² (验证集): 0.9175

--- (C) 交叉验证评估 (Best Lasso) ---


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.2s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.2s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.2s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    5.2s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    4.5s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   6 out of   6 | elapsed:    4.5s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Don

6折交叉验证平均 R²: 0.9179

--- Best Lasso 模型性能已记录 ---
                   OLS  Lasso (alpha=0.0001)
MAE_in    4.130239e+05         414791.843506
RMSE_in   9.383783e+05         948672.536536
MAE_out            NaN         412672.411985
RMSE_out           NaN         951565.625634
R²_in     9.199472e-01              0.919151
R²_out   -3.570738e+05              0.917461
R²_cv    -1.128964e+25              0.917868


In [44]:
# --- 步骤 15.2：使用 RidgeCV 进行超参数调优 ---

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score  # 新增：用于交叉验证R²计算
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd 
import warnings 
import time 

print("--- 步骤 15.2：使用 RidgeCV 进行超参数调优 (寻找最佳 alpha) ---")

# --- 1. 定义 alpha 候选范围 ---
# Ridge 对 alpha 不像 Lasso 那么敏感，我们可以尝试一个更广的范围
alphas_ridge = np.logspace(-1, 3, 10) # 搜索范围 0.1 到 1000
print(f"Ridge 将尝试以下 alpha 值: {alphas_ridge}")

# --- 2. 初始化并运行 RidgeCV ---
ridge_cv_model = RidgeCV(alphas=alphas_ridge, 
                         cv=6, 
                         scoring=None, # 使用默认 R² 评分选择 alpha
                         store_cv_values=False)

print(f"正在运行 RidgeCV (cv=6)...")
start_time = time.time() 
ridge_cv_model.fit(X_train_scaled_df, y_train_log) 
end_time = time.time() 
elapsed_time = end_time - start_time
print(f"\nRidgeCV 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳 alpha ---
best_alpha_ridge = ridge_cv_model.alpha_
print(f"\nRidgeCV 找到的最佳 alpha 是: {best_alpha_ridge:.6f}")

# --- 4. 使用最佳 alpha 模型进行评估 ---
best_ridge_model = ridge_cv_model 

print("\n--- (A) 样本内 (In-sample) 评估 (Best Ridge) ---")
y_train_pred_log_best_ridge = best_ridge_model.predict(X_train_scaled_df)
mae_in_best_ridge, rmse_in_best_ridge = evaluate_model_safe(y_train_log, y_train_pred_log_best_ridge, "Best Ridge In-sample")
# 样本内R²计算（与原始代码整合，明确变量名）
train_r2_ridge = best_ridge_model.score(X_train_scaled_df, y_train_log)
print(f"R² (训练集): {train_r2_ridge:.4f}")

print("\n--- (B) 样本外 (Out-of-sample) 评估 (Best Ridge) ---")
y_val_pred_log_best_ridge = best_ridge_model.predict(X_val_scaled_df) 
mae_out_best_ridge, rmse_out_best_ridge = evaluate_model_safe(y_val_log, y_val_pred_log_best_ridge, "Best Ridge Out-of-sample")
# 样本外R²计算（与原始代码整合，明确变量名）
val_r2_ridge = best_ridge_model.score(X_val_scaled_df, y_val_log)
print(f"R² (验证集): {val_r2_ridge:.4f}")

# --- 新增：交叉验证R²计算（6折，评估模型稳定性） ---
print("\n--- (C) 交叉验证评估 (Best Ridge) ---")
cv_r2_ridge = cross_val_score(
    best_ridge_model, 
    X_train_scaled_df,  # 基于训练特征集进行交叉验证
    y_train_log, 
    cv=6,  # 与调优时一致的6折
    scoring='r2'  # 评估指标为R²
).mean()  # 取6折平均值
print(f"6折交叉验证平均 R²: {cv_r2_ridge:.4f}")

# --- 5. 更新性能记录表（包含R²指标，与其他模型格式统一） ---
try:
    # 移除之前的 Ridge(1.0) 列 (如果存在)
    if 'Ridge (alpha=1.0)' in model_performance.columns:
         model_performance = model_performance.drop(columns=['Ridge (alpha=1.0)'])
    # 添加完整指标（MAE、RMSE、R²）
    model_performance[f'Ridge (alpha={best_alpha_ridge:.2f})'] = [
        mae_in_best_ridge, rmse_in_best_ridge,  # 样本内MAE、RMSE
        mae_out_best_ridge, rmse_out_best_ridge,  # 样本外MAE、RMSE
        train_r2_ridge, val_r2_ridge, cv_r2_ridge  # R²指标（与OLS/Lasso保持一致）
    ]
except NameError:
    # 若model_performance未定义，初始化并添加指标
    print("\n警告：model_performance DataFrame 未定义，正在创建...")
    model_performance = pd.DataFrame(
        index=['MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'R²_in', 'R²_out', 'R²_cv']
    )
    model_performance[f'Ridge (alpha={best_alpha_ridge:.2f})'] = [
        mae_in_best_ridge, rmse_in_best_ridge,
        mae_out_best_ridge, rmse_out_best_ridge,
        train_r2_ridge, val_r2_ridge, cv_r2_ridge
    ]

print("\n--- Best Ridge 模型性能已记录 ---")
print(model_performance)

--- 步骤 15.2：使用 RidgeCV 进行超参数调优 (寻找最佳 alpha) ---
Ridge 将尝试以下 alpha 值: [1.00000000e-01 2.78255940e-01 7.74263683e-01 2.15443469e+00
 5.99484250e+00 1.66810054e+01 4.64158883e+01 1.29154967e+02
 3.59381366e+02 1.00000000e+03]
正在运行 RidgeCV (cv=6)...

RidgeCV 完成。耗时: 115.38 秒

RidgeCV 找到的最佳 alpha 是: 16.681005

--- (A) 样本内 (In-sample) 评估 (Best Ridge) ---
--- Best Ridge In-sample Performance ---
MAE (原始价格): 414,666.45
RMSE (原始价格): 948,942.18
MAE (对数尺度): 0.1767
RMSE (对数尺度): 0.2365
R² (训练集): 0.9192

--- (B) 样本外 (Out-of-sample) 评估 (Best Ridge) ---
--- Best Ridge Out-of-sample Performance ---
MAE (原始价格): 412,913.04
RMSE (原始价格): 951,937.82
MAE (对数尺度): 0.1771
RMSE (对数尺度): 0.2364
R² (验证集): 0.9175

--- (C) 交叉验证评估 (Best Ridge) ---


........................................................................................................................................................................................................................................................................................................................................................................

6折交叉验证平均 R²: 0.9178

--- Best Ridge 模型性能已记录 ---
                   OLS  Lasso (alpha=0.0001)  Ridge (alpha=16.68)
MAE_in    4.130239e+05         414791.843506        414666.445534
RMSE_in   9.383783e+05         948672.536536        948942.178520
MAE_out            NaN         412672.411985        412913.036844
RMSE_out           NaN         951565.625634        951937.820453
R²_in     9.199472e-01              0.919151             0.919176
R²_out   -3.570738e+05              0.917461             0.917461
R²_cv    -1.128964e+25              0.917868             0.917840


In [45]:
# --- 步骤 15.3：使用 ElasticNetCV 进行超参数调优 ---

from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import cross_val_score  # 新增：用于交叉验证R²计算
# (确保 evaluate_model_safe 函数已定义)
import numpy as np
import pandas as pd
import warnings
import time

print("--- 步骤 15.3：使用 ElasticNetCV 进行超参数调优 ---")

# --- 1. 定义 alpha 和 l1_ratio 候选范围 ---
# Alpha 范围 (类似 Lasso，但起点可以稍高一点，因为有 L2)
alphas_enet = np.logspace(-4, -1, 10) # 0.0001 到 0.1

# L1 Ratio 范围 (从接近 Ridge 到接近 Lasso)
l1_ratios_enet = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

print(f"ElasticNet 将尝试 alpha 值 (约): {alphas_enet}")
print(f"ElasticNet 将尝试 l1_ratio 值: {l1_ratios_enet}")

# --- 2. 初始化并运行 ElasticNetCV ---
enet_cv_model = ElasticNetCV(alphas=alphas_enet,
                           l1_ratio=l1_ratios_enet,
                           cv=6, # <-- 作业要求的 6 折
                           random_state=111,
                           n_jobs=4, # <-- 保持较低的 n_jobs
                           max_iter=5000,
                           tol=1e-3, # <-- 保持较宽松的 tol
                           verbose=1) # <-- 保留进度指示

print(f"正在运行 ElasticNetCV (cv=6, n_jobs=4)...")
start_time = time.time()
# --- 使用最终的 drop_first=True, 清理后的数据 ---
enet_cv_model.fit(X_train_scaled_df, y_train_log)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nElasticNetCV 完成。耗时: {elapsed_time:.2f} 秒")

# --- 3. 获取最佳参数 ---
best_alpha_enet = enet_cv_model.alpha_
best_l1_ratio_enet = enet_cv_model.l1_ratio_
print(f"\nElasticNetCV 找到的最佳 alpha 是: {best_alpha_enet:.6f}")
print(f"ElasticNetCV 找到的最佳 l1_ratio 是: {best_l1_ratio_enet:.6f}")

# --- 4. 使用最佳参数模型进行评估 ---
best_enet_model = enet_cv_model
coeffs_best_enet = best_enet_model.coef_
num_non_zero_coeffs_best_enet = np.sum(coeffs_best_enet != 0)
print(f"\n使用最佳参数的 ElasticNet 模型选择了 {num_non_zero_coeffs_best_enet} 个特征 / {X_train_scaled_df.shape[1]}。")

print("\n--- (A) 样本内 (In-sample) 评估 (Best ElasticNet) ---")
y_train_pred_log_best_enet = best_enet_model.predict(X_train_scaled_df)
mae_in_best_enet, rmse_in_best_enet = evaluate_model_safe(y_train_log, y_train_pred_log_best_enet, "Best ElasticNet In-sample")
# 样本内R²计算（统一变量名）
train_r2_enet = best_enet_model.score(X_train_scaled_df, y_train_log)
print(f"R² (训练集): {train_r2_enet:.4f}")

print("\n--- (B) 样本外 (Out-of-sample) 评估 (Best ElasticNet) ---")
y_val_pred_log_best_enet = best_enet_model.predict(X_val_scaled_df)
mae_out_best_enet, rmse_out_best_enet = evaluate_model_safe(y_val_log, y_val_pred_log_best_enet, "Best ElasticNet Out-of-sample")
# 样本外R²计算（统一变量名）
val_r2_enet = best_enet_model.score(X_val_scaled_df, y_val_log)
print(f"R² (验证集): {val_r2_enet:.4f}")

# --- 新增：交叉验证R²计算（6折） ---
print("\n--- (C) 交叉验证评估 (Best ElasticNet) ---")
cv_r2_enet = cross_val_score(
    best_enet_model,
    X_train_scaled_df,  # 基于训练特征集进行交叉验证
    y_train_log,
    cv=6,  # 与调优时一致的6折
    scoring='r2'  # 评估指标为R²
).mean()  # 取6折平均值
print(f"6折交叉验证平均 R²: {cv_r2_enet:.4f}")

# --- 5. 更新性能记录表（包含R²指标，与其他模型格式统一） ---
try:
    model_performance[f'ElasticNet (a={best_alpha_enet:.4f},l1={best_l1_ratio_enet:.2f})'] = [
        mae_in_best_enet, rmse_in_best_enet,  # 样本内MAE、RMSE
        mae_out_best_enet, rmse_out_best_enet,  # 样本外MAE、RMSE
        train_r2_enet, val_r2_enet, cv_r2_enet  # R²指标（与其他模型保持一致）
    ]
except NameError:
    # 若model_performance未定义，初始化并添加指标
    print("\n警告：model_performance DataFrame 未定义，正在创建...")
    model_performance = pd.DataFrame(
        index=['MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'R²_in', 'R²_out', 'R²_cv']
    )
    model_performance[f'ElasticNet (a={best_alpha_enet:.4f},l1={best_l1_ratio_enet:.2f})'] = [
        mae_in_best_enet, rmse_in_best_enet,
        mae_out_best_enet, rmse_out_best_enet,
        train_r2_enet, val_r2_enet, cv_r2_enet
    ]

print("\n--- Best ElasticNet 模型性能已记录 ---")
print(model_performance)

--- 步骤 15.3：使用 ElasticNetCV 进行超参数调优 ---
ElasticNet 将尝试 alpha 值 (约): [0.0001     0.00021544 0.00046416 0.001      0.00215443 0.00464159
 0.01       0.02154435 0.04641589 0.1       ]
ElasticNet 将尝试 l1_ratio 值: [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]
正在运行 ElasticNetCV (cv=6, n_jobs=4)...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
....................................................................................................................................................................................................................................................................................................................................................................................................................................[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:  1.2min finished



ElasticNetCV 完成。耗时: 232.47 秒

ElasticNetCV 找到的最佳 alpha 是: 0.000100
ElasticNetCV 找到的最佳 l1_ratio 是: 1.000000

使用最佳参数的 ElasticNet 模型选择了 655 个特征 / 695。

--- (A) 样本内 (In-sample) 评估 (Best ElasticNet) ---
--- Best ElasticNet In-sample Performance ---
MAE (原始价格): 414,791.84
RMSE (原始价格): 948,672.54
MAE (对数尺度): 0.1768
RMSE (对数尺度): 0.2365
R² (训练集): 0.9192

--- (B) 样本外 (Out-of-sample) 评估 (Best ElasticNet) ---
--- Best ElasticNet Out-of-sample Performance ---
MAE (原始价格): 412,672.41
RMSE (原始价格): 951,565.63
MAE (对数尺度): 0.1771
RMSE (对数尺度): 0.2364
R² (验证集): 0.9175

--- (C) 交叉验证评估 (Best ElasticNet) ---


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   22.3s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   21.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   21.3s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   21.8s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   23.6s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 out of  42 | elapsed:   22.8s finished


6折交叉验证平均 R²: 0.9179

--- Best ElasticNet 模型性能已记录 ---
                   OLS  Lasso (alpha=0.0001)  Ridge (alpha=16.68)  \
MAE_in    4.130239e+05         414791.843506        414666.445534   
RMSE_in   9.383783e+05         948672.536536        948942.178520   
MAE_out            NaN         412672.411985        412913.036844   
RMSE_out           NaN         951565.625634        951937.820453   
R²_in     9.199472e-01              0.919151             0.919176   
R²_out   -3.570738e+05              0.917461             0.917461   
R²_cv    -1.128964e+25              0.917868             0.917840   

          ElasticNet (a=0.0001,l1=1.00)  
MAE_in                    414791.843506  
RMSE_in                   948672.536536  
MAE_out                   412672.411985  
RMSE_out                  951565.625634  
R²_in                          0.919151  
R²_out                         0.917461  
R²_cv                          0.917877  


In [46]:
print("--- 步骤 16：计算 6 折交叉验证 MAE（房价版，基于全流程预处理数据） ---")

# --- 0. 确保全流程预处理后的依赖变量已定义（关键对齐房价版前期步骤） ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
import warnings

# 1. 检查Winsorization后的最终训练特征集（依赖房价版步骤8：缩尾处理）
try:
    X_train_winsorized  # 房价版步骤8生成的“清理+缩尾”后训练特征
except NameError:
    raise NameError("❌ 未找到 X_train_winsorized！请先运行房价版步骤8（Winsorization）生成该变量")

# 2. 检查原始房价训练目标变量（依赖房价版前期数据加载步骤）
try:
    y_train  # 原始房价训练目标（前期数据加载时定义，无_rent后缀）
except NameError:
    raise NameError("❌ 未找到 y_train！请确保房价版前期数据加载步骤已成功运行并定义该变量")

# 3. 重新生成特征缩放数据（基于Winsorized数据，避免缩放不一致问题）
try:
    # 验证缩放数据是否来自房价版Winsorized集（通过列数匹配判断）
    if X_train_scaled_df.shape[1] != X_train_winsorized.shape[1]:
        print("⚠️ 已存在的 X_train_scaled_df 与房价版Winsorized数据列数不匹配，重新生成...")
        raise NameError("缩放数据列数不匹配")
except NameError:
    print("⚠️ X_train_scaled_df 未定义或不匹配，基于房价版 X_train_winsorized 重新生成...")
    
    # 基于房价版步骤8的Winsorized训练集拟合缩放器
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_winsorized)
    # 转换为DataFrame保留列名（与房价版Winsorized集列名完全一致）
    X_train_scaled_df = pd.DataFrame(
        X_train_scaled,
        columns=X_train_winsorized.columns,
        index=X_train_winsorized.index
    )
    print("✅ 已基于房价版 X_train_winsorized 重新生成 X_train_scaled_df")

# 4. 确保房价对数目标变量存在（对齐房价版步骤12的log1p转换逻辑）
try:
    y_train_log  # 房价版对数目标变量（无_rent后缀）
except NameError:
    print("⚠️ 未找到 y_train_log，基于原始房价 y_train 生成...")
    y_train_log = np.log1p(y_train)  # 与房价版步骤12一致的log1p转换（避免0值问题）
    print("✅ 已生成房价版 y_train_log（对数转换后的训练目标）")

# 5. 检查房价版最佳超参数（依赖房价版步骤15：超参数调优）
# Lasso超参数（依赖房价版步骤15.1）
try:
    best_alpha_lasso  # 房价版Lasso最佳alpha（无_rent后缀）
except NameError:
    best_alpha_lasso = 0.0001  # 临时默认值
    print(f"⚠️ 未找到房价版最佳Lasso alpha（步骤15.1未运行），使用临时默认值: {best_alpha_lasso}")

# Ridge超参数（依赖房价版步骤15.2）
try:
    best_alpha_ridge  # 房价版Ridge最佳alpha（无_rent后缀）
except NameError:
    best_alpha_ridge = 16.68  # 临时默认值（参考房价常见调优范围）
    print(f"⚠️ 未找到房价版最佳Ridge alpha（步骤15.2未运行），使用临时默认值: {best_alpha_ridge}")

# ElasticNet超参数（依赖房价版步骤15.3）
try:
    best_alpha_enet, best_l1_ratio_enet  # 房价版ElasticNet最佳参数（无_rent后缀）
except NameError:
    best_alpha_enet = 0.0001    # 临时默认值
    best_l1_ratio_enet = 0.5    # 临时默认值（平衡L1/L2）
    print(f"⚠️ 未找到房价版最佳ElasticNet参数（步骤15.3未运行），使用临时默认值: alpha={best_alpha_enet}, l1_ratio={best_l1_ratio_enet}")


# --- 1. 定义自定义 MAE 评分函数 (房价原始尺度，对齐房价版evaluate_model_safe逻辑) ---
def mae_original_scale_scorer(y_true_log, y_pred_log):
    """自定义评分函数：对数预测→还原房价原始尺度→计算MAE，兼容cross_val_score"""
    try:
        # 与房价版步骤12一致：用expm1还原原始房价（对应log1p转换）
        y_true_orig = np.expm1(y_true_log)  # 原始房价（非租金）
        y_pred_orig = np.expm1(y_pred_log)
        
        # 检查异常值（对齐房价版溢出处理逻辑）
        if not np.all(np.isfinite(y_pred_orig)):
            return -np.inf  # 溢出时返回极差分数
        if (y_pred_orig < 0).any():  # 还原后房价不能为负
            return -np.inf
        
        mae = mean_absolute_error(y_true_orig, y_pred_orig)
        return -mae  # cross_val_score默认“最大化分数”，故返回负MAE
    except (OverflowError, ValueError):
        return -np.inf

# 包装为scorer对象（房价版专用）
mae_scorer = make_scorer(mae_original_scale_scorer, greater_is_better=True)


# --- 2. 准备6折交叉验证（固定随机状态确保房价版结果可复现） ---
kf = KFold(n_splits=6, shuffle=True, random_state=111)  # 与房价版前期交叉验证参数一致
X = X_train_scaled_df  # 房价全流程预处理数据：OHE→清理→Winsorization→缩放
y = y_train_log        # 房价对数转换后的目标变量


# --- 3. 计算各房价模型的6折CV MAE（全基于房价版全流程预处理数据） ---
# 3.1 房价版OLS模型（对齐房价版步骤4.1训练逻辑）
print("\n1/4 正在计算 房价OLS 的 6 折 CV MAE...")
ols_model_for_cv = LinearRegression()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # 忽略缩放/溢出警告
    ols_cv_scores_neg_mae = cross_val_score(
        ols_model_for_cv, X, y, cv=kf, scoring=mae_scorer, n_jobs=4
    )
# 处理无效值（避免房价计算异常）
ols_cv_scores_mae = -ols_cv_scores_neg_mae
valid_ols_scores = ols_cv_scores_mae[np.isfinite(ols_cv_scores_mae)]
ols_cv_mae_mean = np.mean(valid_ols_scores) if len(valid_ols_scores) > 0 else np.nan
# 输出房价OLS结果
print(f"房价OLS 6 折 CV MAE (原始房价，平均值): {ols_cv_mae_mean:,.2f}" 
      if not np.isnan(ols_cv_mae_mean) else "房价OLS 6 折 CV MAE: N/A (计算失败)")
if np.any(np.isinf(ols_cv_scores_mae)) or np.isnan(ols_cv_mae_mean):
    print("  (提示：房价OLS可能因数据分布导致还原原始尺度时溢出)")

# 3.2 房价版Best Lasso模型（对齐房价版步骤15.1逻辑）
print("\n2/4 正在计算 房价Best Lasso 的 6 折 CV MAE...")
best_lasso_for_cv = Lasso(
    alpha=best_alpha_lasso,
    max_iter=10000,  # 提高迭代次数，避免小alpha时不收敛
    tol=1e-3,
    random_state=111
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lasso_cv_scores_neg_mae = cross_val_score(
        best_lasso_for_cv, X, y, cv=kf, scoring=mae_scorer, n_jobs=4
    )
lasso_cv_mae_mean = -np.mean(lasso_cv_scores_neg_mae)
print(f"房价Best Lasso 6 折 CV MAE (原始房价，平均值): {lasso_cv_mae_mean:,.2f}")

# 3.3 房价版Best Ridge模型（对齐房价版步骤15.2逻辑）
print("\n3/4 正在计算 房价Best Ridge 的 6 折 CV MAE...")
best_ridge_for_cv = Ridge(
    alpha=best_alpha_ridge,
    random_state=111
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ridge_cv_scores_neg_mae = cross_val_score(
        best_ridge_for_cv, X, y, cv=kf, scoring=mae_scorer, n_jobs=4
    )
ridge_cv_mae_mean = -np.mean(ridge_cv_scores_neg_mae)
print(f"房价Best Ridge 6 折 CV MAE (原始房价，平均值): {ridge_cv_mae_mean:,.2f}")

# 3.4 房价版Best ElasticNet模型（对齐房价版步骤15.3逻辑）
print("\n4/4 正在计算 房价Best ElasticNet 的 6 折 CV MAE...")
best_enet_for_cv = ElasticNet(
    alpha=best_alpha_enet,
    l1_ratio=best_l1_ratio_enet,
    max_iter=10000,  # 避免不收敛
    tol=1e-3,
    random_state=111
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    enet_cv_scores_neg_mae = cross_val_score(
        best_enet_for_cv, X, y, cv=kf, scoring=mae_scorer, n_jobs=4
    )
enet_cv_mae_mean = -np.mean(enet_cv_scores_neg_mae)
print(f"房价Best ElasticNet 6 折 CV MAE (原始房价，平均值): {enet_cv_mae_mean:,.2f}")


# --- 4. 存储房价版CV结果（对齐房价版model_performance格式） ---
# 确保房价版性能记录表存在（无_rent后缀）
try:
    model_performance  # 房价版前期模型性能表（步骤4.1、15.1-15.3生成）
except NameError:
    print("\n⚠️ 未找到房价版 model_performance，创建新的性能记录DataFrame...")
    model_performance = pd.DataFrame(index=['MAE_in', 'RMSE_in', 'MAE_out', 'RMSE_out', 'MAE_cv'])

# 避免房价CV结果重复写入
if 'MAE_cv' in model_performance.index:
    model_performance = model_performance.drop('MAE_cv')

# 写入各房价模型的CV MAE结果（列名无_rent后缀）
cv_results = pd.Series({
    'OLS': ols_cv_mae_mean,
    f'Lasso (alpha={best_alpha_lasso:.4f})': lasso_cv_mae_mean,
    f'Ridge (alpha={best_alpha_ridge:.2f})': ridge_cv_mae_mean,
    f'ElasticNet (a={best_alpha_enet:.4f},l1={best_l1_ratio_enet:.2f})': enet_cv_mae_mean
}, name='MAE_cv')

# 合并到房价版性能表
model_performance = pd.concat([model_performance, cv_results.to_frame().T])

# --- 5. 格式化输出房价版最终结果（修复标题与列数匹配问题） ---
print("\n" + "="*65)
print("--- 房价版各模型 6 折 CV MAE 结果（基于全流程预处理数据） ---")
print("="*65)

# 重构显示格式：转置为“模型名-房价MAE”结构，确保列数与标题匹配
cv_result_display = model_performance.loc['MAE_cv'].to_frame(name='6折CV MAE（原始房价）')
# 格式化输出（保留2位小数，无效值显示N/A）
print(cv_result_display.to_string(
    float_format='{:,.2f}'.format, 
    na_rep='N/A'
))
print("\n注：MAE越小表示房价模型泛化能力越稳定（基于OHE→清理→Winsorization→缩放全流程数据）")

--- 步骤 16：计算 6 折交叉验证 MAE（房价版，基于全流程预处理数据） ---

1/4 正在计算 房价OLS 的 6 折 CV MAE...


/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1
/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1
/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1
/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1
/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1
/tmp/ipykernel_89/4146702005.py:82: RuntimeWarning: overflow encountered in expm1


房价OLS 6 折 CV MAE: N/A (计算失败)
  (提示：房价OLS可能因数据分布导致还原原始尺度时溢出)

2/4 正在计算 房价Best Lasso 的 6 折 CV MAE...
房价Best Lasso 6 折 CV MAE (原始房价，平均值): 419,806.46

3/4 正在计算 房价Best Ridge 的 6 折 CV MAE...
房价Best Ridge 6 折 CV MAE (原始房价，平均值): 420,025.84

4/4 正在计算 房价Best ElasticNet 的 6 折 CV MAE...
房价Best ElasticNet 6 折 CV MAE (原始房价，平均值): 419,806.46

--- 房价版各模型 6 折 CV MAE 结果（基于全流程预处理数据） ---
                               6折CV MAE（原始房价）
OLS                                       N/A
Lasso (alpha=0.0001)               419,806.46
Ridge (alpha=16.68)                420,025.84
ElasticNet (a=0.0001,l1=1.00)      419,806.46

注：MAE越小表示房价模型泛化能力越稳定（基于OHE→清理→Winsorization→缩放全流程数据）


In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib  # 用于加载保存的缩放器

print("--- 步骤 17：生成预测结果文件 (prediction.csv) ---")

# --- 1. 核心依赖变量检查与修复（确保适配前序流程） ---
print("\n1/5 检查核心依赖变量...")
try:
    # 1.1 检查缩尾后的测试集（修正：前序流程缩尾后测试集名为 X_test_winsorized）
    if 'X_test_winsorized' not in locals():
        raise NameError("X_test_winsorized 未定义！请先运行缩尾处理步骤生成该数据集")
    X_test_final = X_test_winsorized  # 统一测试集变量名，便于后续使用
    print(f"✅ 缩尾后测试集存在（维度：{X_test_final.shape}）")

    # 1.2 检查最佳模型（默认保留Lasso，若用OLS可替换为 ols_model，此处增加兼容性提示）
    if 'best_lasso_model' not in locals():
        # 兼容OLS模型（若未训练Lasso，尝试使用OLS模型）
        if 'ols_model' in locals():
            best_model = ols_model
            print("⚠️ 未找到 best_lasso_model，将使用 OLS 模型（ols_model）进行预测")
        else:
            raise NameError("best_lasso_model 和 ols_model 均未定义！请先运行模型训练步骤")
    else:
        best_model = best_lasso_model
        print(f"✅ 最佳模型存在（模型类型：{type(best_model).__name__}）")

    # 1.3 检查缩放器（优先用已定义的scaler，若不存在则加载保存的缩放器）
    if 'scaler' not in locals():
        try:
            # 加载前序步骤保存的缩放器（步骤：缩尾后的数据缩放 中保存的文件）
            scaler = joblib.load('linear_model_scaler.pkl')
            print("✅ 已加载保存的缩放器（linear_model_scaler.pkl）")
        except FileNotFoundError:
            raise NameError("scaler 未定义且未找到 linear_model_scaler.pkl！请先运行特征缩放步骤")
    print("✅ 所有核心依赖变量检查通过")

except NameError as e:
    print(f"❌ 依赖检查失败：{e}")
    print("⚠️ 请按顺序运行前序步骤（数据清洗→缩尾→特征缩放→模型训练）后再执行")
    raise  # 终止执行，避免后续连锁错误


# --- 2. 测试集数据缩放（与训练集保持一致，避免数据泄露） ---
print("\n2/5 测试集数据缩放...")
try:
    # 用训练集拟合的缩放器转换测试集（仅转换，不重新拟合）
    X_test_scaled = scaler.transform(X_test_final)
    
    # 转换为DataFrame并保留列名和索引（确保与原始测试集ID对应）
    X_test_scaled_df = pd.DataFrame(
        X_test_scaled,
        columns=X_test_final.columns,
        index=X_test_final.index  # 保留缩尾后的索引，用于后续匹配ID
    )
    
    # 特征补全与顺序对齐（确保与模型训练时的特征一致）
    # 获取模型训练时的特征列表（从模型属性中提取，避免手动输入）
    if hasattr(best_model, 'feature_names_in_'):
        model_feature_names = best_model.feature_names_in_
    else:
        # 兼容无feature_names_in_的模型（如旧版sklearn的LinearRegression）
        raise ValueError("❌ 模型未记录训练时的特征列表，请使用sklearn 1.0+版本训练模型")
    
    # 1. 补全缺失特征（用0填充，避免特征数不匹配）
    missing_features = [col for col in model_feature_names if col not in X_test_scaled_df.columns]
    if missing_features:
        print(f"⚠️ 测试集缺失 {len(missing_features)} 个特征，将用0填充（示例：{missing_features[:3]}...）")
        for col in missing_features:
            X_test_scaled_df[col] = 0
    
    # 2. 按模型训练时的特征顺序重新排序（关键：避免特征顺序混乱导致预测错误）
    X_test_scaled_df = X_test_scaled_df[model_feature_names]
    
    # 验证特征数匹配
    if X_test_scaled_df.shape[1] != len(model_feature_names):
        raise ValueError(
            f"🚨 特征数不匹配！模型期望 {len(model_feature_names)} 列，测试集实际 {X_test_scaled_df.shape[1]} 列"
        )
    
    print(f"✅ 测试集缩放完成（维度：{X_test_scaled_df.shape}）")

except Exception as e:
    print(f"❌ 测试集缩放失败：{e}")
    raise


# --- 3. 加载原始测试集并匹配ID（确保预测结果与样本ID对应） ---
print("\n3/5 加载原始测试集并匹配ID...")
# 原始测试集路径（与前序数据加载步骤保持一致）
original_test_path = '/home/mw/input/midata1852/ruc_Class25Q2_test_price (1).csv'
try:
    # 加载原始测试集（仅需ID列，无需其他预处理）
    df_test_original = pd.read_csv(original_test_path)
    
    # 确认ID列存在（若实际列名不同，可修改 id_col 变量，如 '样本ID'/'id'）
    id_col = 'ID'
    if id_col not in df_test_original.columns:
        raise KeyError(f"❌ 原始测试集未找到 '{id_col}' 列！请检查列名并修改代码中 'id_col' 变量")
    
    # 确保原始测试集与处理后的测试集样本数一致（避免样本遗漏）
    if len(df_test_original) != len(X_test_scaled_df):
        raise ValueError(
            f"🚨 样本数不匹配！原始测试集 {len(df_test_original)} 行，处理后测试集 {len(X_test_scaled_df)} 行"
        )
    
    # 提取ID并与处理后的测试集对齐（关键：用原始索引匹配，确保ID不混乱）
    test_ids = df_test_original.set_index(X_test_scaled_df.index)[id_col]  # 按处理后的索引对齐ID
    print(f"✅ 成功匹配 {len(test_ids)} 个样本ID（无重复）")

except FileNotFoundError:
    print(f"❌ 未找到原始测试集文件：{original_test_path}")
    print("⚠️ 请检查文件路径是否正确，或修改代码中 'original_test_path' 变量")
    raise
except KeyError as e:
    print(e)
    raise
except ValueError as e:
    print(e)
    print("⚠️ 请检查测试集预处理步骤是否遗漏样本（如删除行未同步更新ID）")
    raise


# --- 4. 模型预测与结果转换（从对数尺度还原为原始房价） ---
print("\n4/5 模型预测与结果转换...")
try:
    # 1. 对数尺度预测（与模型训练目标一致）
    y_test_pred_log = best_model.predict(X_test_scaled_df)
    print(f"✅ 对数尺度预测完成（预测结果数量：{len(y_test_pred_log)}）")
    
    # 2. 还原为原始房价尺度（对应训练集的 log1p 转换，用 expm1 逆运算）
    y_test_pred_orig = np.expm1(y_test_pred_log)
    
    # 3. 异常值处理（保险措施：避免极端负值或无效值）
    invalid_count = 0
    # 处理负值（理论上 expm1 不会产生负值，此处为极端情况兜底）
    if (y_test_pred_orig < 0).sum() > 0:
        invalid_count += (y_test_pred_orig < 0).sum()
        y_test_pred_orig[y_test_pred_orig < 0] = 0
    # 处理NaN/Inf（数值溢出或模型异常导致）
    if not np.all(np.isfinite(y_test_pred_orig)):
        invalid_count += np.count_nonzero(~np.isfinite(y_test_pred_orig))
        y_test_pred_orig = np.nan_to_num(y_test_pred_orig, nan=0, posinf=0, neginf=0)
    
    if invalid_count > 0:
        print(f"⚠️ 已处理 {invalid_count} 个无效预测值（负值/NaN/Inf），强制设为0")
    
    # 预览预测结果（便于快速验证合理性）
    print(f"\n预测价格示例（前5个）：{y_test_pred_orig[:5].round(2)}")

except Exception as e:
    print(f"❌ 预测过程失败：{e}")
    raise


# --- 5. 生成并保存 prediction.csv（核心输出步骤） ---
print("\n5/5 生成预测结果文件...")
try:
    # 构建预测结果DataFrame（ID列+Price列，符合通用提交格式）
    submission_df = pd.DataFrame({
        id_col: test_ids.values,  # 对齐后的样本ID
        'Price': y_test_pred_orig.round(2)  # 房价保留2位小数，符合实际业务场景
    })
    
    # 最终校验（确保无空值、无重复ID）
    if submission_df.isnull().sum().sum() > 0:
        raise ValueError(f"❌ 预测结果中存在 {submission_df.isnull().sum().sum()} 个空值！")
    if submission_df[id_col].duplicated().sum() > 0:
        raise ValueError(f"❌ 预测结果中存在 {submission_df[id_col].duplicated().sum()} 个重复ID！")
    
    # 保存为CSV文件（不保留索引，编码为utf-8，避免中文乱码）
    submission_filename = 'prediction.csv'
    submission_df.to_csv(submission_filename, index=False, encoding='utf-8')
    
    # 输出最终成功信息
    print(f"\n🎉 预测结果文件生成成功！")
    print(f"📁 文件名：{submission_filename}")
    print(f"📊 文件规格：{submission_df.shape[0]} 行 × {submission_df.shape[1]} 列（{id_col} + Price）")
    print(f"🔍 文件预览：")
    print(submission_df.head(3))

except Exception as e:
    print(f"❌ 保存预测文件失败：{e}")
    raise

print("\n--- 步骤 17 执行完成 ---")

--- 步骤 17：生成预测结果文件 (prediction.csv) ---

1/5 检查核心依赖变量...
✅ 缩尾后测试集存在（维度：(34017, 695)）
✅ 最佳模型存在（模型类型：LassoCV）
✅ 所有核心依赖变量检查通过

2/5 测试集数据缩放...
✅ 测试集缩放完成（维度：(34017, 695)）

3/5 加载原始测试集并匹配ID...


/tmp/ipykernel_89/14925820.py:95: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_original = pd.read_csv(original_test_path)


✅ 成功匹配 34017 个样本ID（无重复）

4/5 模型预测与结果转换...
✅ 对数尺度预测完成（预测结果数量：34017）

预测价格示例（前5个）：[19053712.3   2925502.43  3917970.38  3202770.69 11027185.11]

5/5 生成预测结果文件...

🎉 预测结果文件生成成功！
📁 文件名：prediction.csv
📊 文件规格：34017 行 × 2 列（ID + Price）
🔍 文件预览：
        ID        Price
0  1000000  19053712.30
1  1000001   2925502.43
2  1000002   3917970.38

--- 步骤 17 执行完成 ---
